In [1]:
"""
Graph Sensitivity Experiment — Causal SHAP on Soil SPI Regression (High-Fidelity)
=================================================================================

Tests 11 causal graph variants against the true soil DGP graph and measures
attribution quality via Pearson r, Spearman r, Cosine similarity, RMSE,
and MAE vs ground-truth gamma_i, while saving granular feature attributions.

Graph Categories (Soil Focus):
------------------------------
G00          — True base DAG (OC->pH->ZN structural core, all 7 direct to SPI)
G01–G08      — Spurious inter-feature structural pathways added (1–4 wrong edges)
G09–G10      — Key nodes (Organic Carbon / pH) disconnected from Target SPI

Outputs:
--------
    soil_graph_sensitivity_results.csv   — Comprehensive metrics & feature logs
    soil_graph_sensitivity_bar.png       — Pearson r comparison visualization
    soil_graph_sensitivity_heatmap.png   — Comprehensive matrix footprint
"""

import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
from typing import Dict, List, Set, Tuple

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr, spearmanr
import networkx as nx

try:
    from xgboost import XGBRegressor
    HAVE_XGB = True
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    HAVE_XGB = False

#  CONFIG - Upgraded for extreme high-fidelity stability
CONFIG = {
    'DATA_PATH':         '../../datasets/data_complete.csv',
    'GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    'TEST_SIZE':         0.20,

    # High-Fidelity configurations
    'MAX_PERM':    500,    # Increased path permissions
    'M_RESAMPLES': 128,    # Expanded MC scenario sweeps
    'GLOBAL_K':    100,    # Testing broad data target blocks

    'RANDOM_SEED': 42,
}

FEATURE_NAMES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity'
]

COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])


#  GAMMA COMPUTATION (Model Agnostic Reference Framework)
def compute_gamma(ground_truth_path: str) -> Dict[str, float]:
    """Derive analytic baseline weights from exact continuous contribution parameters."""
    gt = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    total = mean_abs.sum()
    return (mean_abs / total).to_dict()


def build_dag(inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    """Build structural feature-only node matrix from selected edge variations."""
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected in configuration graph: {inter_edges}")
    return G


# Vectorized linear-gaussian sem engine
class LinearMechanismsBatch:
    """Optimized structural engine executing fully vectorized parallel MC calls."""
    def __init__(self, dag: nx.DiGraph):
        self.dag = dag
        self.mech: Dict[str, dict] = {}
        self.topo: List[str] = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def sample_intervention_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        """Matrix execution generation step pulling all M samples concurrently."""
        X_samples = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated = {}

        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var] = np.full(M, val)
                continue

            m = self.mech[var]
            sigma = m['sigma']

            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    if p in simulated:
                        p_matrix[:, p_idx] = simulated[p]
                    else:
                        p_matrix[:, p_idx] = float(x_obs[p])
                mu = m['intercept'] + p_matrix @ m['coef']

            if sigma < 1e-8:
                vals = mu
            else:
                vals = rng.normal(mu, sigma, size=M)

            X_samples[:, v_idx] = vals
            simulated[var] = vals

        return X_samples


# Causal shap batch regressor
class CausalSHAPRegressorBatch:
    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float], missing_spi: List[str], X_train: pd.DataFrame):
        self.model = model
        self.features = list(X_train.columns)
        self._Ef = None

        # Mask weights for features explicitly unlinked from the target node, then re-normalize
        eff = {f: (0.0 if f in missing_spi else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff

        self.sem = LinearMechanismsBatch(dag).fit(X_train)
        self._Ef = float(model.predict(X_train.values).mean())

    def _vc_batch(self, S: Set[str], x: pd.Series, M: int) -> float:
        X_mat = self.sem.sample_intervention_batch(S, x, M)
        return float(np.mean(self.model.predict(X_mat)))

    def explain_instance(self, x: pd.Series, n_perms: int, M: int) -> pd.Series:
        x_row = x.values.reshape(1, -1)
        fx = float(self.model.predict(x_row)[0])

        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc_batch(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc_batch(S_next, x, M)
                phi[self.features.index(feat)] += (self.gamma[feat] * (v_curr - v_prev))
                v_prev, S = v_curr, S_next
        phi /= n_perms

        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, k: int, n_perms: int, M: int) -> pd.Series:
        k = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        phi_list = []
        for t, i in enumerate(idx, 1):
            if t % 20 == 0:
                print(f"      instance {t}/{k} completed...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], n_perms, M))
        phi_mat = pd.concat(phi_list, axis=1)
        return phi_mat.abs().mean(axis=1).rename('Causal_SHAP_AbsMean')


# Metrics and plotting pipelines
def compute_metrics(gamma: Dict[str, float], global_abs: pd.Series) -> Tuple[Dict[str, float], Dict[str, float]]:
    gt_vec = np.array([gamma[f] for f in FEATURE_NAMES])
    raw = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pearson_r, _ = pearsonr(shap_vec, gt_vec)
    spearman_r, _ = spearmanr(shap_vec, gt_vec)
    cosine = float(np.dot(shap_vec, gt_vec) / (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae = float(np.mean(np.abs(shap_vec - gt_vec)))

    metrics = {
        'Pearson_r':  round(float(pearson_r),  4),
        'Spearman_r': round(float(spearman_r), 4),
        'Cosine':     round(cosine,            4),
        'RMSE':       round(rmse,              4),
        'MAE':        round(mae,               4),
    }
    attributions = {f"Attr_{f}": round(shap_vec[i], 4) for i, f in enumerate(FEATURE_NAMES)}
    return metrics, attributions


def plot_bar_chart(results_df: pd.DataFrame, output_path: str):
    fig, ax = plt.subplots(figsize=(11, 7))
    true_pr = results_df.loc[results_df['Label'] == 'G00: True graph', 'Pearson_r'].iloc[0]
    colors = [CAT_COLORS[c] for c in results_df['Category']]
    bars = ax.barh(results_df['Label'], results_df['Pearson_r'], color=colors, edgecolor='white', height=0.65)

    for bar, val in zip(bars, results_df['Pearson_r']):
        ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2, f'{val:.4f}', va='center', ha='left', fontsize=9.5)

    ax.axvline(true_pr, color='#1D9E75', linewidth=2, linestyle='--', alpha=0.8)
    ax.set_xlabel('Pearson r (Normalised Causal SHAP AbsMean vs Soil GT γᵢ)', fontsize=11)
    ax.set_xlim(0, 1.15)
    ax.set_title('Soil Graph Sensitivity: Pearson r Stability Across 11 Variant Configurations', fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


def plot_heatmap(results_df: pd.DataFrame, output_path: str):
    metrics = ['Pearson_r', 'Spearman_r', 'Cosine', 'RMSE']
    data = results_df[metrics].values.T
    fig, ax = plt.subplots(figsize=(14, 4.5))
    im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=data.min(), vmax=1.0)
    ax.set_xticks(range(len(results_df)))
    ax.set_xticklabels(results_df['Label'], rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(len(metrics)))
    ax.set_yticklabels(metrics, fontsize=10)

    for i in range(len(metrics)):
        for j in range(len(results_df)):
            val = data[i, j]
            text_col = 'white' if (metrics[i] == 'RMSE' and val > 0.18) else 'black'
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8.5, color=text_col, fontweight='bold')

    plt.colorbar(im, ax=ax)
    ax.set_title('Soil Variant Structural Evaluation Footprint Heatmap', fontsize=11, fontweight='bold')
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.close(fig)


# Graph configuration variant dictionary (soil system specific)
GRAPH_VARIANTS = [
    dict(label='G00: True graph', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc')], missing_spi=[], category='baseline', description='True Causal Core Structure: OC->pH->ZN, all 7 features -> SPI'),
    dict(label='G01: +N→P', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus')], missing_spi=[], category='extra_edge', description='Add 1 spurious macro-nutrient edge: Nitrogen → Phosphorus'),
    dict(label='G02: +K→N', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Potassium', 'Nitrogen')], missing_spi=[], category='extra_edge', description='Add 1 spurious nutrient edge: Potassium → Nitrogen'),
    dict(label='G03: +EC→ZN', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Electrical Conductivity', 'Zinc')], missing_spi=[], category='extra_edge', description='Add 1 spurious conditional path: Electrical Conductivity → Zinc'),
    dict(label='G04: +P→pH', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Phosphorus', 'pH')], missing_spi=[], category='extra_edge', description='Add 1 wrong feedback edge: Phosphorus → pH'),
    dict(label='G05: +N→P, +K→N', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Potassium', 'Nitrogen')], missing_spi=[], category='extra_edge', description='Add 2 spurious cross-nutrient edges: N→P and K→N'),
    dict(label='G06: N→P→K chain', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Phosphorus', 'Potassium')], missing_spi=[], category='chain', description='Impose 3-node localized nutrient chain structure: N→P→K'),
    dict(label='G07: Full macro chain', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Organic Carbon', 'Nitrogen'), ('Nitrogen', 'Phosphorus'), ('Phosphorus', 'Potassium')], missing_spi=[], category='chain', description='Impose extended structural macro-chain link: OC→N→P→K'),
    dict(label='G08: Dense (4 wrong paths)', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Potassium', 'Nitrogen'), ('Electrical Conductivity', 'pH'), ('Electrical Conductivity', 'Zinc')], missing_spi=[], category='extra_edge', description='Dense distortion framework: 4 spurious structural pathways added'),
    dict(label='G09: No OC→SPI', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc')], missing_spi=['Organic Carbon'], category='missing', description='Organic Carbon disconnected from Target SPI node (feature isolated)'),
    dict(label='G10: No pH→SPI', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc')], missing_spi=['pH'], category='missing', description='pH factor completely unlinked from Target SPI node (feature isolated)'),
]

CAT_COLORS = {'baseline': '#1D9E75', 'extra_edge': '#378ADD', 'chain': '#7F77DD', 'missing': '#D85A30'}


# Main execution
def main():
    print("\n" + "=" * 80)
    print("HIGH-FIDELITY SOIL SPI REGRESSION SENSITIVITY FRAMEWORK")
    print("=" * 80)
    print(f"  MAX_PERM       : {CONFIG['MAX_PERM']}")
    print(f"  M_RESAMPLES    : {CONFIG['M_RESAMPLES']}")
    print(f"  GLOBAL_K       : {CONFIG['GLOBAL_K']}\n")

    try:
        dc = pd.read_csv(CONFIG['DATA_PATH']).rename(columns=COL_MAP)
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['DATA_PATH']}' verification failed."); return

    X = dc[FEATURE_NAMES]
    y = dc['SPI_CONTINUOUS']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True
    )

    try:
        gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'])
    except FileNotFoundError:
        print(f"ERROR: '{CONFIG['GROUND_TRUTH_PATH']}' verification failed."); return

    print("Soil Baseline Target Ground Truth Parameters (γ):")
    for f, v in gamma.items():
        print(f"  {f:<30}  γ = {v:.5f}")

    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES, index=X_test.index)

    print("\nTraining predictive model pipeline...")
    if HAVE_XGB:
        model = XGBRegressor(
            n_estimators=160, max_depth=5, learning_rate=0.08,
            subsample=0.9, colsample_bytree=0.9,
            random_state=CONFIG['RANDOM_SEED'], verbosity=0,
            n_jobs=1, tree_method='hist'
        )
        model_name = "XGBRegressor"
    else:
        model = RandomForestRegressor(n_estimators=160, max_depth=6, random_state=CONFIG['RANDOM_SEED'], n_jobs=1)
        model_name = "RandomForestRegressor"

    model.fit(Xtr.values, y_train.values)
    print(f"  {model_name} trained successfully. R² = {r2_score(y_test.values, model.predict(Xte.values)):.4f}")

    all_results = []
    t_total = time.time()

    for i, g in enumerate(GRAPH_VARIANTS):
        print(f"\n[{i+1:02d}/{len(GRAPH_VARIANTS)}] Evaluating graph matrix {g['label']} ...")
        t0 = time.time()

        dag = build_dag(g['inter_edges'])
        cshap = CausalSHAPRegressorBatch(
            model=model, dag=dag, gamma=gamma, missing_spi=g['missing_spi'], X_train=Xtr
        )
        global_abs = cshap.explain_global(
            Xte, k=CONFIG['GLOBAL_K'], n_perms=CONFIG['MAX_PERM'], M=CONFIG['M_RESAMPLES']
        )

        m, attributions = compute_metrics(gamma, global_abs)
        elapsed = time.time() - t0
        print(f"         Pearson={m['Pearson_r']} | RMSE={m['RMSE']} | Execution Time={elapsed:.1f}s")

        all_results.append({
            'Label':    g['label'],
            'Category': g['category'],
            **m,
            **attributions,
            'Time_s':   round(elapsed, 1),
        })

    df = pd.DataFrame(all_results)
    df.to_csv('soil_graph_sensitivity_results.csv', index=False)

    print("\n" + "=" * 90)
    print("FINAL SOIL REGRESSION GRAPH MATRIX RUN SUMMARY")
    print("=" * 90)
    display_cols = ['Label', 'Pearson_r', 'RMSE', 'Attr_Organic Carbon', 'Attr_pH', 'Attr_Zinc', 'Attr_Nitrogen']
    print(df[display_cols].to_string(index=False))

    plot_bar_chart(df, 'soil_graph_sensitivity_bar.png')
    plot_heatmap(df,   'soil_graph_sensitivity_heatmap.png')
    print(f"\nProcessing complete. Total execution runtime: {(time.time() - t_total)/60:.1f} minutes.")


if __name__ == '__main__':
    main()


HIGH-FIDELITY SOIL SPI REGRESSION SENSITIVITY FRAMEWORK
  MAX_PERM       : 500
  M_RESAMPLES    : 128
  GLOBAL_K       : 100

Soil Baseline Target Ground Truth Parameters (γ):
  Organic Carbon                  γ = 0.06217
  Nitrogen                        γ = 0.40116
  Phosphorus                      γ = 0.07619
  Potassium                       γ = 0.25877
  pH                              γ = 0.08861
  Zinc                            γ = 0.01470
  Electrical Conductivity         γ = 0.09840

Training predictive model pipeline...
  XGBRegressor trained successfully. R² = 0.9882

[01/11] Evaluating graph matrix G00: True graph ...
      instance 20/100 completed...
      instance 40/100 completed...
      instance 60/100 completed...
      instance 80/100 completed...
      instance 100/100 completed...
         Pearson=0.959 | RMSE=0.0625 | Execution Time=454.9s

[02/11] Evaluating graph matrix G01: +N→P ...
      instance 20/100 completed...
      instance 40/100 completed...
      

In [2]:
"""
Focused Ablation Test — Disconnecting Nitrogen from SPI Node
============================================================
Evaluates a single target ablation variant against the Soil framework
under the exact high-fidelity parameters: MAX_PERM=500, M_RESAMPLES=128, GLOBAL_K=100.
"""

import time
import numpy as np
import pandas as pd
import warnings
from typing import Dict, List, Set, Tuple
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import networkx as nx
from xgboost import XGBRegressor

#  EXACT FRAMEWORK CONFIG (High-Fidelity Match)
CONFIG = {
    'DATA_PATH':         '../../datasets/data_complete.csv',
    'GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    'TEST_SIZE':         0.20,
    'MAX_PERM':          500,
    'M_RESAMPLES':       128,
    'GLOBAL_K':          100,
    'RANDOM_SEED':       42,
}

FEATURE_NAMES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity'
]

COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

#  CORE IMPLEMENTATION PIPELINES (Vectorized Match)
def compute_gamma(ground_truth_path: str) -> Dict[str, float]:
    gt = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    return (mean_abs / mean_abs.sum()).to_dict()

class LinearMechanismsBatch:
    def __init__(self, dag: nx.DiGraph):
        self.dag = dag
        self.mech: Dict[str, dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents': [], 'intercept': float(X[var].mean()), 'coef': None, 'sigma': float(X[var].std(ddof=0))}
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents': parents, 'intercept': float(lr.intercept_), 'coef': lr.coef_.astype(float), 'sigma': float(np.std(resid, ddof=0))}
        return self

    def sample_intervention_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        X_samples = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated = {}
        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var] = np.full(M, val)
                continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    p_matrix[:, p_idx] = simulated[p] if p in simulated else float(x_obs[p])
                mu = m['intercept'] + p_matrix @ m['coef']
            X_samples[:, v_idx] = mu if m['sigma'] < 1e-8 else rng.normal(mu, m['sigma'], size=M)
            simulated[var] = X_samples[:, v_idx]
        return X_samples

class CausalSHAPRegressorBatch:
    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float], missing_spi: List[str], X_train: pd.DataFrame):
        self.model = model
        self.features = list(X_train.columns)
        eff = {f: (0.0 if f in missing_spi else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff
        self.sem = LinearMechanismsBatch(dag).fit(X_train)
        self._Ef = float(model.predict(X_train.values).mean())

    def _vc_batch(self, S: Set[str], x: pd.Series, M: int) -> float:
        X_mat = self.sem.sample_intervention_batch(S, x, M)
        return float(np.mean(self.model.predict(X_mat)))

    def explain_instance(self, x: pd.Series, n_perms: int, M: int) -> pd.Series:
        x_row = x.values.reshape(1, -1)
        fx = float(self.model.predict(x_row)[0])
        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc_batch(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc_batch(S_next, x, M)
                phi[self.features.index(feat)] += (self.gamma[feat] * (v_curr - v_prev))
                v_prev, S = v_curr, S_next
        phi /= n_perms
        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, k: int, n_perms: int, M: int) -> pd.Series:
        k = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        phi_list = []
        for t, i in enumerate(idx, 1):
            if t % 20 == 0:
                print(f"      instance {t}/{k} completed...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], n_perms, M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)

# Execution block
def main():
    print("\n" + "=" * 75)
    print("RUNNING ISOLATED TARGET ABLATION CASE: [No Nitrogen -> SPI]")
    print("=" * 75)

    # Load and Split Data
    dc = pd.read_csv(CONFIG['DATA_PATH']).rename(columns=COL_MAP)
    X_train, X_test, y_train, y_test = train_test_split(
        dc[FEATURE_NAMES], dc['SPI_CONTINUOUS'], test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True
    )
    gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'])

    # Scale Data
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES, index=X_test.index)

    # Re-instantiate the precise background XGBoost pipeline
    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9, random_state=CONFIG['RANDOM_SEED'],
        verbosity=0, n_jobs=1, tree_method='hist'
    )
    model.fit(Xtr.values, y_train.values)

    # Build True Base Graph Structure (G00 Core)
    dag = nx.DiGraph()
    dag.add_nodes_from(FEATURE_NAMES)
    dag.add_edge('Organic Carbon', 'pH')
    dag.add_edge('pH', 'Zinc')

    print("Evaluating high-fidelity sample matrix ...")
    t0 = time.time()

    # Execute with Nitrogen explicitly masked out of the target layer array
    cshap = CausalSHAPRegressorBatch(
        model=model, dag=dag, gamma=gamma, missing_spi=['Nitrogen'], X_train=Xtr
    )
    global_abs = cshap.explain_global(
        Xte, k=CONFIG['GLOBAL_K'], n_perms=CONFIG['MAX_PERM'], M=CONFIG['M_RESAMPLES']
    )

    elapsed = time.time() - t0

    # Calculate comparative metrics vector against real ground truth paths
    gt_vec = np.array([gamma[f] for f in FEATURE_NAMES])
    raw = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pearson_r, _ = pearsonr(shap_vec, gt_vec)
    spearman_r, _ = spearmanr(shap_vec, gt_vec)
    cosine = float(np.dot(shap_vec, gt_vec) / (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae = float(np.mean(np.abs(shap_vec - gt_vec)))

    print("\n" + "=" * 75)
    print("TARGET ABLATION EXPERIMENTAL METRICS [No Nitrogen -> SPI]")
    print("=" * 75)
    print(f"  Pearson r  : {pearson_r:.4f}")
    print(f"  Spearman r : {spearman_r:.4f}")
    print(f"  Cosine Sim : {cosine:.4f}")
    print(f"  RMSE       : {rmse:.4f}")
    print(f"  MAE        : {mae:.4f}")
    print(f"  Runtime    : {elapsed:.1f}s")

    print("\nNormalised Feature Attribution Allocations:")
    print("-" * 43)
    for i, f in enumerate(FEATURE_NAMES):
        print(f"  {f:<28} : {shap_vec[i]:.4f}")
    print("=" * 75 + "\n")

if __name__ == '__main__':
    main()


RUNNING ISOLATED TARGET ABLATION CASE: [No Nitrogen -> SPI]
Evaluating high-fidelity sample matrix ...
      instance 20/100 completed...
      instance 40/100 completed...
      instance 60/100 completed...
      instance 80/100 completed...
      instance 100/100 completed...

TARGET ABLATION EXPERIMENTAL METRICS [No Nitrogen -> SPI]
  Pearson r  : 0.2615
  Spearman r : 0.0714
  Cosine Sim : 0.5845
  RMSE       : 0.2013
  MAE        : 0.1271
  Runtime    : 443.9s

Normalised Feature Attribution Allocations:
-------------------------------------------
  Organic Carbon               : 0.1186
  Nitrogen                     : 0.0000
  Phosphorus                   : 0.0562
  Potassium                    : 0.6005
  pH                           : 0.1354
  Zinc                         : 0.0021
  Electrical Conductivity      : 0.0871



In [3]:
"""
Focused Ablation Test — Disconnecting Potassium from SPI Node
==============================================================
Evaluates a single target ablation variant against the Soil framework
under the exact high-fidelity parameters: MAX_PERM=500, M_RESAMPLES=128, GLOBAL_K=100.
"""

import time
import numpy as np
import pandas as pd
import warnings
from typing import Dict, List, Set, Tuple
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import networkx as nx
from xgboost import XGBRegressor

#  EXACT FRAMEWORK CONFIG (High-Fidelity Match)
CONFIG = {
    'DATA_PATH':         '../../datasets/data_complete.csv',
    'GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    'TEST_SIZE':         0.20,
    'MAX_PERM':          500,
    'M_RESAMPLES':       128,
    'GLOBAL_K':          100,
    'RANDOM_SEED':       42,
}

FEATURE_NAMES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity'
]

COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])

#  CORE IMPLEMENTATION PIPELINES (Vectorized Match)
def compute_gamma(ground_truth_path: str) -> Dict[str, float]:
    gt = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    return (mean_abs / mean_abs.sum()).to_dict()

class LinearMechanismsBatch:
    def __init__(self, dag: nx.DiGraph):
        self.dag = dag
        self.mech: Dict[str, dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents': [], 'intercept': float(X[var].mean()), 'coef': None, 'sigma': float(X[var].std(ddof=0))}
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents': parents, 'intercept': float(lr.intercept_), 'coef': lr.coef_.astype(float), 'sigma': float(np.std(resid, ddof=0))}
        return self

    def sample_intervention_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        X_samples = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated = {}
        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var] = np.full(M, val)
                continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    p_matrix[:, p_idx] = simulated[p] if p in simulated else float(x_obs[p])
                mu = m['intercept'] + p_matrix @ m['coef']
            X_samples[:, v_idx] = mu if m['sigma'] < 1e-8 else rng.normal(mu, m['sigma'], size=M)
            simulated[var] = X_samples[:, v_idx]
        return X_samples

class CausalSHAPRegressorBatch:
    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float], missing_spi: List[str], X_train: pd.DataFrame):
        self.model = model
        self.features = list(X_train.columns)
        eff = {f: (0.0 if f in missing_spi else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff
        self.sem = LinearMechanismsBatch(dag).fit(X_train)
        self._Ef = float(model.predict(X_train.values).mean())

    def _vc_batch(self, S: Set[str], x: pd.Series, M: int) -> float:
        X_mat = self.sem.sample_intervention_batch(S, x, M)
        return float(np.mean(self.model.predict(X_mat)))

    def explain_instance(self, x: pd.Series, n_perms: int, M: int) -> pd.Series:
        x_row = x.values.reshape(1, -1)
        fx = float(self.model.predict(x_row)[0])
        phi = np.zeros(len(self.features), dtype=float)
        for _ in range(n_perms):
            perm = rng.permutation(self.features)
            S: Set[str] = set()
            v_prev = self._vc_batch(S, x, M)
            for feat in perm:
                S_next = S | {feat}
                v_curr = self._vc_batch(S_next, x, M)
                phi[self.features.index(feat)] += (self.gamma[feat] * (v_curr - v_prev))
                v_prev, S = v_curr, S_next
        phi /= n_perms
        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame, k: int, n_perms: int, M: int) -> pd.Series:
        k = min(k, len(X_test))
        idx = rng.choice(len(X_test), size=k, replace=False)
        phi_list = []
        for t, i in enumerate(idx, 1):
            if t % 20 == 0:
                print(f"      instance {t}/{k} completed...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], n_perms, M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)

# Execution block
def main():
    print("\n" + "=" * 75)
    print("RUNNING ISOLATED TARGET ABLATION CASE: [No Potassium -> SPI]")
    print("=" * 75)

    # Load and Split Data
    dc = pd.read_csv(CONFIG['DATA_PATH']).rename(columns=COL_MAP)
    X_train, X_test, y_train, y_test = train_test_split(
        dc[FEATURE_NAMES], dc['SPI_CONTINUOUS'], test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True
    )
    gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'])

    # Scale Data
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES, index=X_test.index)

    # Train model
    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9, random_state=CONFIG['RANDOM_SEED'],
        verbosity=0, n_jobs=1, tree_method='hist'
    )
    model.fit(Xtr.values, y_train.values)

    # Build True Base Graph Structure (G00 Core)
    dag = nx.DiGraph()
    dag.add_nodes_from(FEATURE_NAMES)
    dag.add_edge('Organic Carbon', 'pH')
    dag.add_edge('pH', 'Zinc')

    print("Evaluating high-fidelity sample matrix ...")
    t0 = time.time()

    # Execute with Potassium explicitly masked out of the target layer array
    cshap = CausalSHAPRegressorBatch(
        model=model, dag=dag, gamma=gamma, missing_spi=['Potassium'], X_train=Xtr
    )
    global_abs = cshap.explain_global(
        Xte, k=CONFIG['GLOBAL_K'], n_perms=CONFIG['MAX_PERM'], M=CONFIG['M_RESAMPLES']
    )

    elapsed = time.time() - t0

    # Calculate comparative metrics vector against real ground truth paths
    gt_vec = np.array([gamma[f] for f in FEATURE_NAMES])
    raw = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pearson_r, _ = pearsonr(shap_vec, gt_vec)
    spearman_r, _ = spearmanr(shap_vec, gt_vec)
    cosine = float(np.dot(shap_vec, gt_vec) / (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae = float(np.mean(np.abs(shap_vec - gt_vec)))

    print("\n" + "=" * 75)
    print("TARGET ABLATION EXPERIMENTAL METRICS [No Potassium -> SPI]")
    print("=" * 75)
    print(f"  Pearson r  : {pearson_r:.4f}")
    print(f"  Spearman r : {spearman_r:.4f}")
    print(f"  Cosine Sim : {cosine:.4f}")
    print(f"  RMSE       : {rmse:.4f}")
    print(f"  MAE        : {mae:.4f}")
    print(f"  Runtime    : {elapsed:.1f}s")

    print("\nNormalised Feature Attribution Allocations:")
    print("-" * 43)
    for i, f in enumerate(FEATURE_NAMES):
        print(f"  {f:<28} : {shap_vec[i]:.4f}")
    print("=" * 75 + "\n")

if __name__ == '__main__':
    main()


RUNNING ISOLATED TARGET ABLATION CASE: [No Potassium -> SPI]
Evaluating high-fidelity sample matrix ...
      instance 20/100 completed...
      instance 40/100 completed...
      instance 60/100 completed...
      instance 80/100 completed...
      instance 100/100 completed...

TARGET ABLATION EXPERIMENTAL METRICS [No Potassium -> SPI]
  Pearson r  : 0.7497
  Spearman r : 0.2143
  Cosine Sim : 0.8491
  RMSE       : 0.1249
  MAE        : 0.0847
  Runtime    : 441.4s

Normalised Feature Attribution Allocations:
-------------------------------------------
  Organic Carbon               : 0.1456
  Nitrogen                     : 0.5851
  Phosphorus                   : 0.0478
  Potassium                    : 0.0000
  pH                           : 0.1123
  Zinc                         : 0.0055
  Electrical Conductivity      : 0.1039



In [7]:
"""
Exact Causal Shapley Framework — Soil SPI Regression (Fixed Type Casting)
========================================================================
Replaces random permutation sampling with exact mathematical enumeration
across all 128 unique subsets of the 7-feature soil network matrix.

Parameters:
-----------
    M_RESAMPLES : 128 (Exact batch conditional simulations)
    GLOBAL_K    : 1000 (Full test set evaluation for absolute fidelity)
"""

import time
import itertools
import numpy as np
import pandas as pd
import warnings
import math
from typing import Dict, List, Set, Tuple
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import networkx as nx
from xgboost import XGBRegressor

#  CONFIG - Configured for Full Mathematical Exactness
CONFIG = {
    'DATA_PATH':         '../../datasets/data_complete.csv',
    'GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    'TEST_SIZE':         0.20,
    'M_RESAMPLES':       128,    # MC samples per vc(S) call
    'RANDOM_SEED':       42,
}

FEATURE_NAMES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity'
]

COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}

np.random.seed(CONFIG['RANDOM_SEED'])


# Data generation & stratification lines
def compute_gamma(ground_truth_path: str) -> Dict[str, float]:
    gt = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    return (mean_abs / mean_abs.sum()).to_dict()


# Vectorized structural interventional engine
class LinearMechanismsBatch:
    def __init__(self, dag: nx.DiGraph):
        self.dag = dag
        self.mech: Dict[str, dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents': [], 'intercept': float(X[var].mean()), 'coef': None, 'sigma': float(X[var].std(ddof=0))}
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents': parents, 'intercept': float(lr.intercept_), 'coef': lr.coef_.astype(float), 'sigma': float(np.std(resid, ddof=0))}
        return self

    def sample_intervention_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        X_samples = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated = {}
        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var] = np.full(M, val)
                continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    p_matrix[:, p_idx] = simulated[p] if p in simulated else float(x_obs[p])
                mu = m['intercept'] + p_matrix @ m['coef']
            X_samples[:, v_idx] = mu if m['sigma'] < 1e-8 else np.random.normal(mu, m['sigma'], size=M)
            simulated[var] = X_samples[:, v_idx]
        return X_samples


# Exact enumeration causal shap coupler
class ExactCausalSHAPRegressor:
    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float], missing_spi: List[str], X_train: pd.DataFrame):
        self.model = model
        self.features = list(X_train.columns)
        self.n_features = len(self.features)

        # Apply target layer structural masking adjustments
        eff = {f: (0.0 if f in missing_spi else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff

        self.sem = LinearMechanismsBatch(dag).fit(X_train)
        self._Ef = float(model.predict(X_train.values).mean())

        # Pre-compute combinatorial Shapley weights safely via factorials
        self.coef_weights = {}
        for sz in range(self.n_features):
            num = math.factorial(sz) * math.factorial(self.n_features - sz - 1)
            den = math.factorial(self.n_features)
            self.coef_weights[sz] = float(num / den)

    def explain_instance_exact(self, x: pd.Series, M: int) -> pd.Series:
        """Computes exact Causal Shapley values by iterating over all 128 unique subsets."""
        x_row = x.values.reshape(1, -1)
        fx = float(self.model.predict(x_row)[0])
        phi = np.zeros(self.n_features, dtype=float)

        # Cache interventional expected values for all possible subsets to avoid redrawing
        subset_v_cache = {}

        # Generate and evaluate all 128 structural subsets
        for r in range(self.n_features + 1):
            for subset in itertools.combinations(self.features, r):
                S = set(subset)
                X_mat = self.sem.sample_intervention_batch(S, x, M)
                val = float(np.mean(self.model.predict(X_mat)))
                subset_v_cache[frozenset(S)] = val

        # Calculate margined delta steps weighted across analytical combinatorial boundaries
        for idx, feat in enumerate(self.features):
            remaining_feats = [f for f in self.features if f != feat]
            for r in range(self.n_features):
                for subset in itertools.combinations(remaining_feats, r):
                    S = frozenset(subset)
                    S_with_feat = frozenset(set(subset) | {feat})

                    v_without = subset_v_cache[S]
                    v_with = subset_v_cache[S_with_feat]

                    # Exact structural weight application (Eq. 10 Core Modification)
                    weight = self.coef_weights[len(subset)]
                    phi[idx] += weight * self.gamma[feat] * (v_with - v_without)

        # Apply strict property normalisation bounds (Eq. 11 check)
        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global_exact(self, X_test: pd.DataFrame, M: int) -> pd.Series:
        """Processes global matrix across all rows within the evaluation target block."""
        phi_list = []
        k = len(X_test)
        for t in range(k):
            if (t + 1) % 50 == 0 or t == 0 or (t + 1) == k:
                print(f"      [Instance Tracking Log] Finished {t+1}/{k} evaluation profiles...", flush=True)
            phi_list.append(self.explain_instance_exact(X_test.iloc[t], M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Main system forward runner
def main():
    print("\n" + "=" * 80)
    print("INITIALIZING EXACT CAUSAL SHAPLEY HIGH-FIDELITY SUBSET PIPELINE")
    print("=" * 80)
    print(f"  Total Subsets Evaluated per Instance : 128")
    print(f"  M_RESAMPLES (Simulation Scale Block) : {CONFIG['M_RESAMPLES']}")

    # Load and scale full dataset
    dc = pd.read_csv(CONFIG['DATA_PATH']).rename(columns=COL_MAP)
    X = dc[FEATURE_NAMES]
    y = dc['SPI_CONTINUOUS']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True
    )

    # Evaluate across all 1000 items in X_test (~6 seconds per instance block)
    print(f"  GLOBAL_K (Evaluating All Test Rows) : {len(X_test)}")
    print("-" * 80)

    gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'])
    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES, index=X_test.index)

    # Initialize the structural background model
    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9, random_state=CONFIG['RANDOM_SEED'],
        verbosity=0, n_jobs=1, tree_method='hist'
    )
    model.fit(Xtr.values, y_train.values)

    # Define G00 Baseline Structural core
    dag = nx.DiGraph()
    dag.add_nodes_from(FEATURE_NAMES)
    dag.add_edge('Organic Carbon', 'pH')
    dag.add_edge('pH', 'Zinc')

    print("\nBeginning exact calculations over test rows (Expected run length ~37 mins)...")
    t0 = time.time()

    exact_engine = ExactCausalSHAPRegressor(
        model=model, dag=dag, gamma=gamma, missing_spi=[], X_train=Xtr
    )
    global_abs = exact_engine.explain_global_exact(Xte, M=CONFIG['M_RESAMPLES'])

    elapsed = time.time() - t0

    # Compute definitive zero-noise statistical alignment arrays
    gt_vec = np.array([gamma[f] for f in FEATURE_NAMES])
    raw = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pearson_r, _ = pearsonr(shap_vec, gt_vec)
    spearman_r, _ = spearmanr(shap_vec, gt_vec)
    cosine = float(np.dot(shap_vec, gt_vec) / (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae = float(np.mean(np.abs(shap_vec - gt_vec)))

    print("\n" + "=" * 80)
    print("DEFINITIVE EXACT CAUSAL SHAPLEY METRICS MATRIX (ZERO NOISE BENCHMARK)")
    print("=" * 80)
    print(f"  Exact Pearson r  : {pearson_r:.5f}")
    print(f"  Exact Spearman r : {spearman_r:.4f}")
    print(f"  Exact Cosine Sim : {cosine:.4f}")
    print(f"  Exact RMSE       : {rmse:.4f}")
    print(f"  Exact MAE        : {mae:.4f}")
    print(f"  Total Runtime    : {elapsed/60:.1f} minutes")

    print("\nDefinitive Feature Attribution Allocations:")
    print("-" * 43)
    for i, f in enumerate(FEATURE_NAMES):
        print(f"  {f:<28} : {shap_vec[i]:.4f}")
    print("=" * 80 + "\n")

if __name__ == '__main__':
    main()


INITIALIZING EXACT CAUSAL SHAPLEY HIGH-FIDELITY SUBSET PIPELINE
  Total Subsets Evaluated per Instance : 128
  M_RESAMPLES (Simulation Scale Block) : 128
  GLOBAL_K (Evaluating All Test Rows) : 1000
--------------------------------------------------------------------------------

Beginning exact calculations over test rows (Expected run length ~37 mins)...
      [Instance Tracking Log] Finished 1/1000 evaluation profiles...
      [Instance Tracking Log] Finished 50/1000 evaluation profiles...
      [Instance Tracking Log] Finished 100/1000 evaluation profiles...
      [Instance Tracking Log] Finished 150/1000 evaluation profiles...
      [Instance Tracking Log] Finished 200/1000 evaluation profiles...
      [Instance Tracking Log] Finished 250/1000 evaluation profiles...
      [Instance Tracking Log] Finished 300/1000 evaluation profiles...
      [Instance Tracking Log] Finished 350/1000 evaluation profiles...
      [Instance Tracking Log] Finished 400/1000 evaluation profiles...
    

In [8]:
"""
Definitive Exact Causal Shapley Framework Matrix — Structural Variations (G00-G08)
================================================================================
Replaces random permutation approximations with exact mathematical power-set
enumeration (128 subsets) across the core structural soil causal graphs.

Logs definitive metrics alongside explicit feature attribution pathways.

Outputs:
--------
    exact_soil_sensitivity_results.csv  — Zero-noise structural metrics spreadsheet
"""

import time
import itertools
import numpy as np
import pandas as pd
import warnings
import math
from typing import Dict, List, Set, Tuple
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import networkx as nx
from xgboost import XGBRegressor

#  CONFIG - High-Fidelity Exact Combinatorial Bounds
CONFIG = {
    'DATA_PATH':         '../../datasets/data_complete.csv',
    'GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    'TEST_SIZE':         0.20,
    'M_RESAMPLES':       128,    # MC simulations per active subset block

    # Target Evaluation Scope Tracker
    # Evaluates 100 highly representative records for a stable ~30-min run.
    'EXACT_K':           100,

    'RANDOM_SEED':       42,
}

FEATURE_NAMES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity'
]

COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])


# Analytic reference framework
def compute_gamma(ground_truth_path: str) -> Dict[str, float]:
    gt = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    return (mean_abs / mean_abs.sum()).to_dict()


def build_dag(inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected in layout specifications: {inter_edges}")
    return G


# Vectorized structural engine
class LinearMechanismsBatch:
    def __init__(self, dag: nx.DiGraph):
        self.dag = dag
        self.mech: Dict[str, dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents': [], 'intercept': float(X[var].mean()), 'coef': None, 'sigma': float(X[var].std(ddof=0))}
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents': parents, 'intercept': float(lr.intercept_), 'coef': lr.coef_.astype(float), 'sigma': float(np.std(resid, ddof=0))}
        return self

    def sample_intervention_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        X_samples = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated = {}
        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var] = np.full(M, val)
                continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    p_matrix[:, p_idx] = simulated[p] if p in simulated else float(x_obs[p])
                mu = m['intercept'] + p_matrix @ m['coef']
            X_samples[:, v_idx] = mu if m['sigma'] < 1e-8 else rng.normal(mu, m['sigma'], size=M)
            simulated[var] = X_samples[:, v_idx]
        return X_samples


# Exact enumeration engine
class ExactCausalSHAPRegressor:
    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float], missing_spi: List[str], X_train: pd.DataFrame):
        self.model = model
        self.features = list(X_train.columns)
        self.n_features = len(self.features)

        eff = {f: (0.0 if f in missing_spi else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff

        self.sem = LinearMechanismsBatch(dag).fit(X_train)
        self._Ef = float(model.predict(X_train.values).mean())

        self.coef_weights = {}
        for sz in range(self.n_features):
            num = math.factorial(sz) * math.factorial(self.n_features - sz - 1)
            den = math.factorial(self.n_features)
            self.coef_weights[sz] = float(num / den)

    def explain_instance_exact(self, x: pd.Series, M: int) -> pd.Series:
        x_row = x.values.reshape(1, -1)
        fx = float(self.model.predict(x_row)[0])
        phi = np.zeros(self.n_features, dtype=float)

        subset_v_cache = {}

        # Evaluate 128 power set targets
        for r in range(self.n_features + 1):
            for subset in itertools.combinations(self.features, r):
                S = set(subset)
                X_mat = self.sem.sample_intervention_batch(S, x, M)
                subset_v_cache[frozenset(S)] = float(np.mean(self.model.predict(X_mat)))

        # Analytical combination transitions
        for idx, feat in enumerate(self.features):
            remaining_feats = [f for f in self.features if f != feat]
            for r in range(self.n_features):
                for subset in itertools.combinations(remaining_feats, r):
                    S = frozenset(subset)
                    S_with_feat = frozenset(set(subset) | {feat})

                    weight = self.coef_weights[len(subset)]
                    phi[idx] += weight * self.gamma[feat] * (subset_v_cache[S_with_feat] - subset_v_cache[S])

        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global_exact(self, X_test: pd.DataFrame, k: int, M: int) -> pd.Series:
        phi_list = []
        k = min(k, len(X_test))
        idx = np.arange(k)
        for idx_count, i in enumerate(idx):
            if (idx_count + 1) % 20 == 0:
                print(f"      instance {idx_count + 1}/{k} completed...", flush=True)
            phi_list.append(self.explain_instance_exact(X_test.iloc[i], M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Core structural soil graph configurations (g00 - g08)
GRAPH_VARIANTS = [
    dict(label='G00: True graph', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc')], missing_spi=[]),
    dict(label='G01: +N→P', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus')], missing_spi=[]),
    dict(label='G02: +K→N', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Potassium', 'Nitrogen')], missing_spi=[]),
    dict(label='G03: +EC→ZN', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Electrical Conductivity', 'Zinc')], missing_spi=[]),
    dict(label='G04: +P→pH', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Phosphorus', 'pH')], missing_spi=[]),
    dict(label='G05: +N→P, +K→N', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Potassium', 'Nitrogen')], missing_spi=[]),
    dict(label='G06: N→P→K chain', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Phosphorus', 'Potassium')], missing_spi=[]),
    dict(label='G07: Full macro chain', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Organic Carbon', 'Nitrogen'), ('Nitrogen', 'Phosphorus'), ('Phosphorus', 'Potassium')], missing_spi=[]),
    dict(label='G08: Dense (4 wrong paths)', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Potassium', 'Nitrogen'), ('Electrical Conductivity', 'pH'), ('Electrical Conductivity', 'Zinc')], missing_spi=[]),
]

# Main loop pipeline execution
def main():
    print("\n" + "=" * 90)
    print("LAUNCHING STRUCTURAL MULTI-DAG EXACT CAUSAL SHAPLEY SOIL MATRIX")
    print("=" * 90)
    print(f"  Exact Combinatorial Subsets Evaluated per Instance : 128")
    print(f"  Global Row Instances Processed per DAG Matrix (K)  : {CONFIG['EXACT_K']}")
    print(f"  Total Graphs to Map sequentially                   : {len(GRAPH_VARIANTS)}\n")

    dc = pd.read_csv(CONFIG['DATA_PATH']).rename(columns=COL_MAP)
    X_train, X_test, y_train, y_test = train_test_split(
        dc[FEATURE_NAMES], dc['SPI_CONTINUOUS'], test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True
    )
    gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'])

    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES, index=X_test.index)

    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9,
        random_state=CONFIG['RANDOM_SEED'], verbosity=0, n_jobs=1, tree_method='hist'
    )
    model.fit(Xtr.values, y_train.values)

    all_results = []
    t_total = time.time()

    # Loop exactly through the structural configurations (G00 to G08)
    for idx, g in enumerate(GRAPH_VARIANTS, 1):
        print(f"\n[{idx:02d}/{len(GRAPH_VARIANTS)}] Evaluating EXACT calculations for matrix {g['label']} ...")
        t0 = time.time()

        dag = build_dag(g['inter_edges'])
        exact_engine = ExactCausalSHAPRegressor(
            model=model, dag=dag, gamma=gamma, missing_spi=g['missing_spi'], X_train=Xtr
        )
        global_abs = exact_engine.explain_global_exact(Xte, k=CONFIG['EXACT_K'], M=CONFIG['M_RESAMPLES'])

        elapsed = time.time() - t0

        # Calculate definitive zero-noise arrays
        gt_vec = np.array([gamma[f] for f in FEATURE_NAMES])
        raw = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
        shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

        pearson_r, _ = pearsonr(shap_vec, gt_vec)
        spearman_r, _ = spearmanr(shap_vec, gt_vec)
        cosine = float(np.dot(shap_vec, gt_vec) / (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
        rmse = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
        mae = float(np.mean(np.abs(shap_vec - gt_vec)))

        print(f"         Pearson={pearson_r:.4f} | RMSE={rmse:.4f} | Complete in {elapsed:.1f}s")

        attributions = {f"Attr_{f}": round(shap_vec[i], 4) for i, f in enumerate(FEATURE_NAMES)}
        all_results.append({
            'Label':     g['label'],
            'Pearson_r': round(float(pearson_r), 4),
            'RMSE':      round(float(rmse), 4),
            **attributions,
            'Time_s':    round(elapsed, 1)
        })

    # Save to disk
    df = pd.DataFrame(all_results)
    df.to_csv('exact_soil_sensitivity_results.csv', index=False)

    print("\n" + "=" * 95)
    print("DEFINITIVE ZERO-NOISE EXPERIMENTAL MATRIX RESULTS (G00-G08)")
    print("=" * 95)
    display_cols = ['Label', 'Pearson_r', 'RMSE', 'Attr_Nitrogen', 'Attr_Potassium', 'Attr_Organic Carbon', 'Attr_pH']
    print(df[display_cols].to_string(index=False))
    print(f"\nProcessing matrix completely finished in {(time.time() - t_total)/60:.1f} minutes.\n")

if __name__ == '__main__':
    main()


LAUNCHING STRUCTURAL MULTI-DAG EXACT CAUSAL SHAPLEY SOIL MATRIX
  Exact Combinatorial Subsets Evaluated per Instance : 128
  Global Row Instances Processed per DAG Matrix (K)  : 100
  Total Graphs to Map sequentially                   : 9


[01/9] Evaluating EXACT calculations for matrix G00: True graph ...
      instance 20/100 completed...
      instance 40/100 completed...
      instance 60/100 completed...
      instance 80/100 completed...
      instance 100/100 completed...
         Pearson=0.9846 | RMSE=0.0629 | Complete in 14.6s

[02/9] Evaluating EXACT calculations for matrix G01: +N→P ...
      instance 20/100 completed...
      instance 40/100 completed...
      instance 60/100 completed...
      instance 80/100 completed...
      instance 100/100 completed...
         Pearson=0.9867 | RMSE=0.0646 | Complete in 13.7s

[03/9] Evaluating EXACT calculations for matrix G02: +K→N ...
      instance 20/100 completed...
      instance 40/100 completed...
      instance 60/100 comp

In [9]:
"""
Definitive Exact Causal Shapley Framework Matrix — Structural Variations (G00-G08)
================================================================================
Replaces random permutation approximations with exact mathematical power-set
enumeration (128 subsets) across the core structural soil causal graphs.

Logs definitive metrics (Pearson, Spearman, Cosine, RMSE) alongside attribution pathways.

Outputs:
--------
    exact_soil_sensitivity_results.csv  — Zero-noise structural metrics spreadsheet
"""

import time
import itertools
import numpy as np
import pandas as pd
import warnings
import math
from typing import Dict, List, Set, Tuple
from scipy.stats import pearsonr, spearmanr

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import networkx as nx
from xgboost import XGBRegressor

#  CONFIG - High-Fidelity Exact Combinatorial Bounds
CONFIG = {
    'DATA_PATH':         '../../datasets/data_complete.csv',
    'GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    'TEST_SIZE':         0.20,
    'M_RESAMPLES':       128,    # MC simulations per active subset block

    # Target Evaluation Scope Tracker
    # Evaluates 100 highly representative records for a stable ~30-min run.
    'EXACT_K':           100,

    'RANDOM_SEED':       42,
}

FEATURE_NAMES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity'
]

COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])


# Analytic reference framework
def compute_gamma(ground_truth_path: str) -> Dict[str, float]:
    gt = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':           gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                 gt['N_base']   - gt['N_penalty'],
        'Phosphorus':               gt['P_base']   - gt['P_penalty'],
        'Potassium':                gt['K_base']   - gt['K_penalty'],
        'pH':                      -gt['PH_penalty'],
        'Zinc':                     gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    return (mean_abs / mean_abs.sum()).to_dict()


def build_dag(inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for (u, v) in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected in layout specifications: {inter_edges}")
    return G


# Vectorized structural engine
class LinearMechanismsBatch:
    def __init__(self, dag: nx.DiGraph):
        self.dag = dag
        self.mech: Dict[str, dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {'parents': [], 'intercept': float(X[var].mean()), 'coef': None, 'sigma': float(X[var].std(ddof=0))}
            else:
                lr = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {'parents': parents, 'intercept': float(lr.intercept_), 'coef': lr.coef_.astype(float), 'sigma': float(np.std(resid, ddof=0))}
        return self

    def sample_intervention_batch(self, S: Set[str], x_obs: pd.Series, M: int) -> np.ndarray:
        X_samples = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated = {}
        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var] = np.full(M, val)
                continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    p_matrix[:, p_idx] = simulated[p] if p in simulated else float(x_obs[p])
                mu = m['intercept'] + p_matrix @ m['coef']
            X_samples[:, v_idx] = mu if m['sigma'] < 1e-8 else rng.normal(mu, m['sigma'], size=M)
            simulated[var] = X_samples[:, v_idx]
        return X_samples


# Exact enumeration engine
class ExactCausalSHAPRegressor:
    def __init__(self, model, dag: nx.DiGraph, gamma: Dict[str, float], missing_spi: List[str], X_train: pd.DataFrame):
        self.model = model
        self.features = list(X_train.columns)
        self.n_features = len(self.features)

        eff = {f: (0.0 if f in missing_spi else gamma[f]) for f in self.features}
        tot = sum(eff.values())
        self.gamma = {f: v / tot for f, v in eff.items()} if tot > 1e-12 else eff

        self.sem = LinearMechanismsBatch(dag).fit(X_train)
        self._Ef = float(model.predict(X_train.values).mean())

        self.coef_weights = {}
        for sz in range(self.n_features):
            num = math.factorial(sz) * math.factorial(self.n_features - sz - 1)
            den = math.factorial(self.n_features)
            self.coef_weights[sz] = float(num / den)

    def explain_instance_exact(self, x: pd.Series, M: int) -> pd.Series:
        x_row = x.values.reshape(1, -1)
        fx = float(self.model.predict(x_row)[0])
        phi = np.zeros(self.n_features, dtype=float)

        subset_v_cache = {}

        # Evaluate 128 power set targets
        for r in range(self.n_features + 1):
            for subset in itertools.combinations(self.features, r):
                S = set(subset)
                X_mat = self.sem.sample_intervention_batch(S, x, M)
                subset_v_cache[frozenset(S)] = float(np.mean(self.model.predict(X_mat)))

        # Analytical combination transitions
        for idx, feat in enumerate(self.features):
            remaining_feats = [f for f in self.features if f != feat]
            for r in range(self.n_features):
                for subset in itertools.combinations(remaining_feats, r):
                    S = frozenset(subset)
                    S_with_feat = frozenset(set(subset) | {feat})

                    weight = self.coef_weights[len(subset)]
                    phi[idx] += weight * self.gamma[feat] * (subset_v_cache[S_with_feat] - subset_v_cache[S])

        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global_exact(self, X_test: pd.DataFrame, k: int, M: int) -> pd.Series:
        phi_list = []
        k = min(k, len(X_test))
        idx = np.arange(k)
        for idx_count, i in enumerate(idx):
            if (idx_count + 1) % 20 == 0:
                print(f"      instance {idx_count + 1}/{k} completed...", flush=True)
            phi_list.append(self.explain_instance_exact(X_test.iloc[i], M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Core structural soil graph configurations (g00 - g08)
GRAPH_VARIANTS = [
    dict(label='G00: True graph', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc')], missing_spi=[]),
    dict(label='G01: +N→P', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus')], missing_spi=[]),
    dict(label='G02: +K→N', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Potassium', 'Nitrogen')], missing_spi=[]),
    dict(label='G03: +EC→ZN', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Electrical Conductivity', 'Zinc')], missing_spi=[]),
    dict(label='G04: +P→pH', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Phosphorus', 'pH')], missing_spi=[]),
    dict(label='G05: +N→P, +K→N', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Potassium', 'Nitrogen')], missing_spi=[]),
    dict(label='G06: N→P→K chain', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Phosphorus', 'Potassium')], missing_spi=[]),
    dict(label='G07: Full macro chain', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Organic Carbon', 'Nitrogen'), ('Nitrogen', 'Phosphorus'), ('Phosphorus', 'Potassium')], missing_spi=[]),
    dict(label='G08: Dense (4 wrong paths)', inter_edges=[('Organic Carbon', 'pH'), ('pH', 'Zinc'), ('Nitrogen', 'Phosphorus'), ('Potassium', 'Nitrogen'), ('Electrical Conductivity', 'pH'), ('Electrical Conductivity', 'Zinc')], missing_spi=[]),
]

# Main loop pipeline execution
def main():
    print("\n" + "=" * 90)
    print("LAUNCHING STRUCTURAL MULTI-DAG EXACT CAUSAL SHAPLEY SOIL MATRIX")
    print("=" * 90)
    print(f"  Exact Combinatorial Subsets Evaluated per Instance : 128")
    print(f"  Global Row Instances Processed per DAG Matrix (K)  : {CONFIG['EXACT_K']}")
    print(f"  Total Graphs to Map sequentially                   : {len(GRAPH_VARIANTS)}\n")

    dc = pd.read_csv(CONFIG['DATA_PATH']).rename(columns=COL_MAP)
    X_train, X_test, y_train, y_test = train_test_split(
        dc[FEATURE_NAMES], dc['SPI_CONTINUOUS'], test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['RANDOM_SEED'], shuffle=True
    )
    gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'])

    scaler = StandardScaler()
    Xtr = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURE_NAMES, index=X_train.index)
    Xte = pd.DataFrame(scaler.transform(X_test), columns=FEATURE_NAMES, index=X_test.index)

    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08, subsample=0.9, colsample_bytree=0.9,
        random_state=CONFIG['RANDOM_SEED'], verbosity=0, n_jobs=1, tree_method='hist'
    )
    model.fit(Xtr.values, y_train.values)

    all_results = []
    t_total = time.time()

    for idx, g in enumerate(GRAPH_VARIANTS, 1):
        print(f"\n[{idx:02d}/{len(GRAPH_VARIANTS)}] Evaluating EXACT calculations for matrix {g['label']} ...")
        t0 = time.time()

        dag = build_dag(g['inter_edges'])
        exact_engine = ExactCausalSHAPRegressor(
            model=model, dag=dag, gamma=gamma, missing_spi=g['missing_spi'], X_train=Xtr
        )
        global_abs = exact_engine.explain_global_exact(Xte, k=CONFIG['EXACT_K'], M=CONFIG['M_RESAMPLES'])

        elapsed = time.time() - t0

        gt_vec = np.array([gamma[f] for f in FEATURE_NAMES])
        raw = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
        shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

        pearson_r, _ = pearsonr(shap_vec, gt_vec)
        spearman_r, _ = spearmanr(shap_vec, gt_vec)
        cosine = float(np.dot(shap_vec, gt_vec) / (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
        rmse = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))

        print(f"         Pearson={pearson_r:.4f} | Spearman={spearman_r:.4f} | Cosine={cosine:.4f} | RMSE={rmse:.4f} | Complete in {elapsed:.1f}s")

        attributions = {f"Attr_{f}": round(shap_vec[i], 4) for i, f in enumerate(FEATURE_NAMES)}

        all_results.append({
            'Label':      g['label'],
            'Pearson_r':  round(float(pearson_r), 4),
            'Spearman_r': round(float(spearman_r), 4),
            'Cosine':     round(float(cosine), 4),
            'RMSE':       round(float(rmse), 4),
            **attributions,
            'Time_s':     round(elapsed, 1)
        })

    df = pd.DataFrame(all_results)
    df.to_csv('exact_soil_sensitivity_results.csv', index=False)

    print("\n" + "=" * 115)
    print("DEFINITIVE ZERO-NOISE EXPERIMENTAL MATRIX RESULTS (G00-G08)")
    print("=" * 115)

    display_cols = ['Label', 'Pearson_r', 'Spearman_r', 'Cosine', 'RMSE', 'Attr_Nitrogen', 'Attr_Potassium']
    print(df[display_cols].to_string(index=False))
    print(f"\nProcessing matrix completely finished in {(time.time() - t_total)/60:.1f} minutes.\n")

if __name__ == '__main__':
    main()


LAUNCHING STRUCTURAL MULTI-DAG EXACT CAUSAL SHAPLEY SOIL MATRIX
  Exact Combinatorial Subsets Evaluated per Instance : 128
  Global Row Instances Processed per DAG Matrix (K)  : 100
  Total Graphs to Map sequentially                   : 9


[01/9] Evaluating EXACT calculations for matrix G00: True graph ...
      instance 20/100 completed...
      instance 40/100 completed...
      instance 60/100 completed...
      instance 80/100 completed...
      instance 100/100 completed...
         Pearson=0.9846 | Spearman=0.8214 | Cosine=0.9750 | RMSE=0.0629 | Complete in 14.2s

[02/9] Evaluating EXACT calculations for matrix G01: +N→P ...
      instance 20/100 completed...
      instance 40/100 completed...
      instance 60/100 completed...
      instance 80/100 completed...
      instance 100/100 completed...
         Pearson=0.9867 | Spearman=0.8214 | Cosine=0.9748 | RMSE=0.0646 | Complete in 14.4s

[03/9] Evaluating EXACT calculations for matrix G02: +K→N ...
      instance 20/100 comple

In [1]:
"""
Edge Weight Sensitivity Experiment — Causal SHAP on Soil Regression
====================================================================
Fixes the TRUE causal graph (OC→pH, pH→Zn) across all variants and
systematically perturbs only the SEM edge weights (β coefficients),
measuring how weight misspecification alone degrades attribution quality.

Complements the graph-structure sensitivity experiments (G00–G08) which
fixed weights and varied topology; here topology is fixed and weights vary.

Variants W00–W14:
  W00                 : baseline — exact data-fitted weights (reference)
  W01, W02            : scale-up   (×2, ×5)
  W03, W04            : scale-down (×0.5, ×0.1)
  W05                 : zero OC→pH weight only
  W06                 : zero pH→Zn weight only
  W07                 : negate OC→pH weight
  W08                 : negate pH→Zn weight
  W09, W10            : Gaussian noise on weights (σ=0.5, σ=2.0)
  W11                 : both edges ×2
  W12                 : both edges ×0.1
  W13                 : OC→pH negated + pH→Zn ×0.1
  W14                 : both edges zeroed (degenerates to star DAG)

Outputs:
--------
    edge_weight_sensitivity_results.csv   — metrics + attributions per variant
"""

import time
import itertools
import math
import warnings
from copy import deepcopy
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Config
CONFIG = {
    'DATA_PATH':         '../../datasets/data_complete.csv',
    'GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    'TEST_SIZE':         0.20,
    'M_RESAMPLES':       128,   # Monte Carlo samples per subset evaluation
    'EXACT_K':           100,   # Test instances evaluated per variant
    'RANDOM_SEED':       42,
}

FEATURE_NAMES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity'
]

COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}

TRUE_EDGES = [('Organic Carbon', 'pH'), ('pH', 'Zinc')]

np.random.seed(CONFIG['RANDOM_SEED'])
rng = np.random.default_rng(CONFIG['RANDOM_SEED'])


# Ground truth gamma
def compute_gamma(ground_truth_path: str) -> Dict[str, float]:
    gt = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':          gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                gt['N_base']   - gt['N_penalty'],
        'Phosphorus':              gt['P_base']   - gt['P_penalty'],
        'Potassium':               gt['K_base']   - gt['K_penalty'],
        'pH':                     -gt['PH_penalty'],
        'Zinc':                    gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    return (mean_abs / mean_abs.sum()).to_dict()


def build_dag(inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(FEATURE_NAMES)
    for u, v in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected: {inter_edges}")
    return G


class LinearMechanismsBatch:
    """
    Fits a linear-Gaussian SEM on the given DAG, then exposes
    `patch_weights()` so individual edge coefficients can be overridden
    without re-fitting — isolating the effect of weight misspecification.
    """

    def __init__(self, dag: nx.DiGraph):
        self.dag  = dag
        self.mech: Dict[str, dict] = {}
        self.topo = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in FEATURE_NAMES:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float).copy(),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def patch_weights(self, patches: Dict[Tuple[str, str], float]) -> 'LinearMechanismsBatch':
        """
        Override specific edge weights in the fitted SEM without re-fitting.

        patches: {(parent, child): new_coefficient_value}
                 e.g. {('Organic Carbon', 'pH'): 0.0}

        Returns self so calls can be chained.
        """
        for (parent, child), new_val in patches.items():
            if child not in self.mech:
                raise KeyError(f"No mechanism fitted for child node '{child}'")
            m = self.mech[child]
            if parent not in m['parents']:
                raise KeyError(f"'{parent}' is not a fitted parent of '{child}'")
            idx = m['parents'].index(parent)
            m['coef'][idx] = float(new_val)
        return self

    def sample_intervention_batch(
        self, S: Set[str], x_obs: pd.Series, M: int
    ) -> np.ndarray:
        X_samples   = np.empty((M, len(FEATURE_NAMES)))
        feat_to_idx = {f: i for i, f in enumerate(FEATURE_NAMES)}
        simulated   = {}

        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_samples[:, v_idx] = val
                simulated[var]      = np.full(M, val)
                continue

            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                p_matrix = np.empty((M, len(m['parents'])))
                for p_idx, p in enumerate(m['parents']):
                    p_matrix[:, p_idx] = (
                        simulated[p] if p in simulated else float(x_obs[p])
                    )
                mu = m['intercept'] + p_matrix @ m['coef']

            X_samples[:, v_idx] = (
                mu if m['sigma'] < 1e-8
                else rng.normal(mu, m['sigma'], size=M)
            )
            simulated[var] = X_samples[:, v_idx]

        return X_samples


#  EXACT CAUSAL SHAP ENGINE  (identical to G00-G08 script)
class ExactCausalSHAPRegressor:
    def __init__(
        self,
        model,
        sem: LinearMechanismsBatch,
        gamma: Dict[str, float],
        X_train: pd.DataFrame,
    ):
        self.model     = model
        self.features  = list(X_train.columns)
        self.n_features = len(self.features)
        self.gamma     = gamma
        self.sem       = sem
        self._Ef       = float(model.predict(X_train.values).mean())

        # Precompute Shapley kernel weights
        self.coef_weights = {}
        for sz in range(self.n_features):
            num = math.factorial(sz) * math.factorial(self.n_features - sz - 1)
            den = math.factorial(self.n_features)
            self.coef_weights[sz] = float(num / den)

    def explain_instance_exact(self, x: pd.Series, M: int) -> pd.Series:
        x_row = x.values.reshape(1, -1)
        fx    = float(self.model.predict(x_row)[0])
        phi   = np.zeros(self.n_features, dtype=float)

        subset_v_cache = {}
        for r in range(self.n_features + 1):
            for subset in itertools.combinations(self.features, r):
                S      = frozenset(subset)
                X_mat  = self.sem.sample_intervention_batch(S, x, M)
                subset_v_cache[S] = float(np.mean(self.model.predict(X_mat)))

        for idx, feat in enumerate(self.features):
            remaining = [f for f in self.features if f != feat]
            for r in range(self.n_features):
                for subset in itertools.combinations(remaining, r):
                    S          = frozenset(subset)
                    S_with     = frozenset(set(subset) | {feat})
                    weight     = self.coef_weights[len(subset)]
                    phi[idx]  += weight * self.gamma[feat] * (
                        subset_v_cache[S_with] - subset_v_cache[S]
                    )

        # Rescale to sum to f(x) - E[f]
        s = float(np.sum(phi))
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)

        return pd.Series(phi, index=self.features)

    def explain_global_exact(
        self, X_test: pd.DataFrame, k: int, M: int
    ) -> pd.Series:
        k        = min(k, len(X_test))
        phi_list = []
        for count in range(k):
            if (count + 1) % 20 == 0:
                print(f"      instance {count+1}/{k} ...", flush=True)
            phi_list.append(self.explain_instance_exact(X_test.iloc[count], M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Weight perturbation variant definitions  (w00-w14)
def make_variants(fitted_sem: LinearMechanismsBatch) -> List[dict]:
    """
    Returns a list of variant dicts. Each dict contains:
      label   : display name
      patches : Dict[(parent, child) -> new_value]  — empty = baseline

    The fitted_sem is used to read the true coefficients so scaling
    factors are applied relative to the actual data-fitted values.
    """
    # Convenience: read the true fitted coefficients
    b_OC_pH = float(
        fitted_sem.mech['pH']['coef'][
            fitted_sem.mech['pH']['parents'].index('Organic Carbon')
        ]
    )
    b_pH_Zn = float(
        fitted_sem.mech['Zinc']['coef'][
            fitted_sem.mech['Zinc']['parents'].index('pH')
        ]
    )

    print(f"\n  True fitted weights:")
    print(f"    OC → pH  β = {b_OC_pH:.6f}")
    print(f"    pH → Zn  β = {b_pH_Zn:.6f}\n")

    noise_seed = np.random.default_rng(0)

    variants = [
        # ── Baseline ──────────────────────────────────────────────
        dict(label='W00: baseline (true weights)', patches={}),

        # ── Scale-up: single edge ─────────────────────────────────
        dict(label='W01: OC→pH ×2',
             patches={('Organic Carbon', 'pH'): b_OC_pH * 2}),
        dict(label='W02: OC→pH ×5',
             patches={('Organic Carbon', 'pH'): b_OC_pH * 5}),
        dict(label='W03: pH→Zn ×2',
             patches={('pH', 'Zinc'): b_pH_Zn * 2}),
        dict(label='W04: pH→Zn ×5',
             patches={('pH', 'Zinc'): b_pH_Zn * 5}),

        # ── Scale-down: single edge ───────────────────────────────
        dict(label='W05: OC→pH ×0.5',
             patches={('Organic Carbon', 'pH'): b_OC_pH * 0.5}),
        dict(label='W06: OC→pH ×0.1',
             patches={('Organic Carbon', 'pH'): b_OC_pH * 0.1}),
        dict(label='W07: pH→Zn ×0.5',
             patches={('pH', 'Zinc'): b_pH_Zn * 0.5}),
        dict(label='W08: pH→Zn ×0.1',
             patches={('pH', 'Zinc'): b_pH_Zn * 0.1}),

        # ── Zero / edge suppression ───────────────────────────────
        dict(label='W09: OC→pH = 0',
             patches={('Organic Carbon', 'pH'): 0.0}),
        dict(label='W10: pH→Zn = 0',
             patches={('pH', 'Zinc'): 0.0}),

        # ── Negation ──────────────────────────────────────────────
        dict(label='W11: OC→pH negated',
             patches={('Organic Carbon', 'pH'): -b_OC_pH}),
        dict(label='W12: pH→Zn negated',
             patches={('pH', 'Zinc'): -b_pH_Zn}),

        # ── Gaussian noise on weights ─────────────────────────────
        dict(label='W13: OC→pH + N(0,0.5)',
             patches={('Organic Carbon', 'pH'): b_OC_pH + float(noise_seed.normal(0, 0.5))}),
        dict(label='W13b: pH→Zn + N(0,0.5)',
             patches={('pH', 'Zinc'): b_pH_Zn + float(noise_seed.normal(0, 0.5))}),
        dict(label='W14: OC→pH + N(0,2.0)',
             patches={('Organic Carbon', 'pH'): b_OC_pH + float(noise_seed.normal(0, 2.0))}),
        dict(label='W14b: pH→Zn + N(0,2.0)',
             patches={('pH', 'Zinc'): b_pH_Zn + float(noise_seed.normal(0, 2.0))}),

        # ── Compound: both edges perturbed simultaneously ─────────
        dict(label='W15: both edges ×2',
             patches={('Organic Carbon', 'pH'): b_OC_pH * 2,
                      ('pH', 'Zinc'):           b_pH_Zn * 2}),
        dict(label='W16: both edges ×0.1',
             patches={('Organic Carbon', 'pH'): b_OC_pH * 0.1,
                      ('pH', 'Zinc'):           b_pH_Zn * 0.1}),
        dict(label='W17: OC→pH neg. + pH→Zn ×0.1',
             patches={('Organic Carbon', 'pH'): -b_OC_pH,
                      ('pH', 'Zinc'):            b_pH_Zn * 0.1}),
        dict(label='W18: both edges = 0 (star DAG)',
             patches={('Organic Carbon', 'pH'): 0.0,
                      ('pH', 'Zinc'):           0.0}),
    ]
    return variants


# Helpers
def metrics_and_attrs(
    gamma: Dict[str, float], global_abs: pd.Series
) -> Tuple[dict, dict]:
    gt_vec   = np.array([gamma[f]             for f in FEATURE_NAMES])
    raw      = np.array([global_abs.get(f, 0) for f in FEATURE_NAMES])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pearson_r,  _ = pearsonr(shap_vec,  gt_vec)
    spearman_r, _ = spearmanr(shap_vec, gt_vec)
    cosine  = float(
        np.dot(shap_vec, gt_vec) /
        (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12)
    )
    rmse = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae  = float(np.mean(np.abs(shap_vec - gt_vec)))

    metrics = {
        'Pearson_r':  round(float(pearson_r),  4),
        'Spearman_r': round(float(spearman_r), 4),
        'Cosine':     round(cosine,            4),
        'RMSE':       round(rmse,              4),
        'MAE':        round(mae,               4),
    }
    attributions = {
        f"Attr_{f}": round(shap_vec[i], 4)
        for i, f in enumerate(FEATURE_NAMES)
    }
    return metrics, attributions


# Main
def main():
    print("\n" + "=" * 90)
    print("EDGE WEIGHT SENSITIVITY — EXACT CAUSAL SHAPLEY (TRUE GRAPH FIXED)")
    print("=" * 90)
    print(f"  Exact subsets per instance  : 2^7 = 128")
    print(f"  MC samples per subset (M)   : {CONFIG['M_RESAMPLES']}")
    print(f"  Test instances per variant  : {CONFIG['EXACT_K']}")

    dc = pd.read_csv(CONFIG['DATA_PATH']).rename(columns=COL_MAP)
    X_train, X_test, y_train, y_test = train_test_split(
        dc[FEATURE_NAMES], dc['SPI_CONTINUOUS'],
        test_size=CONFIG['TEST_SIZE'],
        random_state=CONFIG['RANDOM_SEED'],
        shuffle=True,
    )
    gamma = compute_gamma(CONFIG['GROUND_TRUTH_PATH'])

    scaler = StandardScaler()
    Xtr = pd.DataFrame(
        scaler.fit_transform(X_train), columns=FEATURE_NAMES, index=X_train.index
    )
    Xte = pd.DataFrame(
        scaler.transform(X_test), columns=FEATURE_NAMES, index=X_test.index
    )

    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        random_state=CONFIG['RANDOM_SEED'],
        verbosity=0, n_jobs=1, tree_method='hist',
    )
    model.fit(Xtr.values, y_train.values)

    dag = build_dag(TRUE_EDGES)
    base_sem = LinearMechanismsBatch(dag).fit(Xtr)

    variants = make_variants(base_sem)
    print(f"  Total variants to evaluate  : {len(variants)}\n")

    all_results = []
    t_total = time.time()

    for v_idx, v in enumerate(variants, 1):
        print(f"\n[{v_idx:02d}/{len(variants)}] {v['label']}")

        # Deep-copy the fitted SEM, then apply weight patches
        sem = deepcopy(base_sem)
        if v['patches']:
            sem.patch_weights(v['patches'])
            for (p, c), val in v['patches'].items():
                print(f"    patched {p} → {c}: {val:.6f}")

        t0 = time.time()
        engine = ExactCausalSHAPRegressor(
            model=model, sem=sem, gamma=gamma, X_train=Xtr
        )
        global_abs = engine.explain_global_exact(
            Xte, k=CONFIG['EXACT_K'], M=CONFIG['M_RESAMPLES']
        )
        elapsed = time.time() - t0

        m, attrs = metrics_and_attrs(gamma, global_abs)
        print(
            f"  Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
            f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f} | "
            f"MAE={m['MAE']:.4f} | {elapsed:.1f}s"
        )

        all_results.append({
            'Label':     v['label'],
            'Patches':   str(v['patches']),
            **m,
            **attrs,
            'Time_s':    round(elapsed, 1),
        })

    df = pd.DataFrame(all_results)
    out_path = 'edge_weight_sensitivity_results.csv'
    df.to_csv(out_path, index=False)

    print("\n" + "=" * 115)
    print("EDGE WEIGHT SENSITIVITY RESULTS (W00–W18)")
    print("=" * 115)
    display_cols = [
        'Label', 'Pearson_r', 'Spearman_r', 'Cosine', 'RMSE', 'MAE',
        'Attr_Organic Carbon', 'Attr_pH', 'Attr_Zinc',
    ]
    print(df[display_cols].to_string(index=False))
    print(f"\nTotal runtime: {(time.time() - t_total) / 60:.1f} minutes")
    print(f"Results saved to: {out_path}\n")


if __name__ == '__main__':
    main()


EDGE WEIGHT SENSITIVITY — EXACT CAUSAL SHAPLEY (TRUE GRAPH FIXED)
  Exact subsets per instance  : 2^7 = 128
  MC samples per subset (M)   : 128
  Test instances per variant  : 100

  True fitted weights:
    OC → pH  β = -0.708307
    pH → Zn  β = -0.194760

  Total variants to evaluate  : 21


[01/21] W00: baseline (true weights)
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  Pearson=0.9846 | Spearman=0.8214 | Cosine=0.9750 | RMSE=0.0629 | MAE=0.0547 | 8.9s

[02/21] W01: OC→pH ×2
    patched Organic Carbon → pH: -1.416613
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ...
      instance 100/100 ...
  Pearson=0.9854 | Spearman=0.7500 | Cosine=0.9757 | RMSE=0.0621 | MAE=0.0533 | 8.9s

[03/21] W02: OC→pH ×5
    patched Organic Carbon → pH: -3.541533
      instance 20/100 ...
      instance 40/100 ...
      instance 60/100 ...
      instance 80/100 ..

In [2]:
"""
Weight Sensitivity Experiment — Causal SHAP on Soil & Yield Regression
=======================================================================
Fixes the TRUE causal graph topology for each dataset and perturbs:

  (A) SEM inter-feature edge weights  (β coefficients in the SEM)
  (B) Feature→target importance weights (γᵢ used inside Causal SHAP)

This complements the graph-structure experiments (G00–G08 / G00–G10)
which varied topology while keeping weights fixed. Here topology is
fixed and weights vary — isolating weight misspecification as the
sole source of attribution error.

DATASETS
--------
  Soil  : data_complete.csv + ground_truth.csv
          True graph edges: OC→pH, pH→Zn
          Features: Organic Carbon, Nitrogen, Phosphorus, Potassium,
                    pH, Zinc, Electrical Conductivity  (7 features)

  Yield : yield_regression_preprocessed.npz
          True graph edges: none (star DAG — all features independent)
          Features: Nitrogen, Irrigation, Precipitation, Herbicide (4 features)

VARIANT FAMILIES  (applied to each dataset independently)
-----------------
  SEM weight variants  — only meaningful where inter-feature edges exist
    W00  : baseline (true fitted weights)
    W01–W04 : scale single edge ×2, ×5, ×0.5, ×0.1
    W05–W06 : zero each edge individually
    W07–W08 : negate each edge individually
    W09–W10 : Gaussian noise σ=0.5 / σ=2.0 on each edge
    W11–W14 : compound (both edges): ×2, ×0.1, neg+shrink, both=0

  Gamma (γᵢ) variants  — feature→target importance misspecification
    G00  : baseline (true γ)
    G01  : uniform γ (all features equally weighted)
    G02  : top-1 dominant (all weight on highest-γ feature)
    G03  : inverted γ (highest true → lowest assigned, and vice versa)
    G04  : scale highest-γ feature ×2 (renormalised)
    G05  : scale lowest-γ feature ×5 (renormalised)
    G06  : zero highest-γ feature (renormalised over rest)
    G07  : zero lowest-γ feature (renormalised over rest)
    G08  : add Gaussian noise σ=0.1 (renormalised, clipped ≥ 0)
    G09  : add Gaussian noise σ=0.3 (renormalised, clipped ≥ 0)

Outputs
-------
    soil_weight_sensitivity_results.csv
    yield_weight_sensitivity_results.csv
"""

import time
import itertools
import math
import warnings
from copy import deepcopy
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Global config
CONFIG = {
    # Soil paths
    'SOIL_DATA_PATH':         '../../datasets/data_complete.csv',
    'SOIL_GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    # Yield path
    'YIELD_DATA_PATH':        '../../datasets/yield_regression_preprocessed.npz',

    'TEST_SIZE':     0.20,
    'M_RESAMPLES':   128,   # MC samples per subset evaluation
    'EXACT_K':       100,   # Test instances evaluated per variant
    'RANDOM_SEED':   42,
}

# ── Soil ──────────────────────────────────────────────────────────
SOIL_FEATURES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity',
]
SOIL_COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}
SOIL_TRUE_EDGES = [('Organic Carbon', 'pH'), ('pH', 'Zinc')]

# ── Yield ─────────────────────────────────────────────────────────
YIELD_FEATURES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']
YIELD_TRUE_EDGES: List[Tuple[str, str]] = []   # star DAG - no inter-feature edges

YIELD_TRANSLOG = {
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI':  0.04, 'betaNP':  0.02, 'betaNH':  0.01,
    'betaIP':  0.03, 'betaIH':  0.01, 'betaPH':  0.005,
}

rng_noise = np.random.default_rng(0)


# Ground-truth gamma functions
def compute_soil_gamma(ground_truth_path: str) -> Dict[str, float]:
    gt  = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':          gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                gt['N_base']   - gt['N_penalty'],
        'Phosphorus':              gt['P_base']   - gt['P_penalty'],
        'Potassium':               gt['K_base']   - gt['K_penalty'],
        'pH':                     -gt['PH_penalty'],
        'Zinc':                    gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    return (mean_abs / mean_abs.sum()).to_dict()


def compute_yield_gamma(X_train: pd.DataFrame) -> Dict[str, float]:
    c   = YIELD_TRANSLOG
    lm  = {f: float(np.log(X_train[f]).mean()) for f in YIELD_FEATURES}
    lnN, lnI, lnP, lnH = (lm['Nitrogen'], lm['Irrigation'],
                           lm['Precipitation'], lm['Herbicide'])
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    total = sum(elasticities.values())
    return {f: v / total for f, v in elasticities.items()}


# Dag builder
def build_dag(features: List[str],
              inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(features)
    for u, v in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected: {inter_edges}")
    return G


class LinearMechanismsBatch:
    """
    Linear-Gaussian SEM fitted on a given DAG.
    `patch_weights()` overrides specific β coefficients in-place
    without re-running OLS — isolating weight misspecification.
    """
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = dag
        self.features = features
        self.mech:    Dict[str, dict] = {}
        self.topo:    List[str] = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float).copy(),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def patch_weights(self,
                      patches: Dict[Tuple[str, str], float]
                      ) -> 'LinearMechanismsBatch':
        """patches: {(parent, child): new_β_value}"""
        for (parent, child), val in patches.items():
            m   = self.mech[child]
            idx = m['parents'].index(parent)
            m['coef'][idx] = float(val)
        return self

    def sample_intervention_batch(
        self, S: Set[str], x_obs: pd.Series, M: int, rng_inst
    ) -> np.ndarray:
        X_out       = np.empty((M, len(self.features)))
        feat_to_idx = {f: i for i, f in enumerate(self.features)}
        simulated   = {}

        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_out[:, v_idx] = val
                simulated[var]  = np.full(M, val)
                continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                pm = np.empty((M, len(m['parents'])))
                for pi, p in enumerate(m['parents']):
                    pm[:, pi] = simulated[p] if p in simulated else float(x_obs[p])
                mu = m['intercept'] + pm @ m['coef']
            X_out[:, v_idx] = (
                mu if m['sigma'] < 1e-8
                else rng_inst.normal(mu, m['sigma'], size=M)
            )
            simulated[var] = X_out[:, v_idx]
        return X_out


# Exact causal shap engine
class ExactCausalSHAPRegressor:
    def __init__(self,
                 model,
                 sem:     LinearMechanismsBatch,
                 gamma:   Dict[str, float],
                 X_train: pd.DataFrame):
        self.model      = model
        self.features   = list(X_train.columns)
        self.n          = len(self.features)
        self.gamma      = gamma        # caller is responsible for normalisation
        self.sem        = sem
        self._Ef        = float(model.predict(X_train.values).mean())
        self._rng       = np.random.default_rng(CONFIG['RANDOM_SEED'])

        # Shapley kernel weights keyed by coalition size
        self._kw = {}
        for sz in range(self.n):
            num = math.factorial(sz) * math.factorial(self.n - sz - 1)
            self._kw[sz] = float(num / math.factorial(self.n))

    def _v(self, S: frozenset, x: pd.Series, M: int) -> float:
        X_mat = self.sem.sample_intervention_batch(S, x, M, self._rng)
        return float(np.mean(self.model.predict(X_mat)))

    def explain_instance(self, x: pd.Series, M: int) -> pd.Series:
        fx  = float(self.model.predict(x.values.reshape(1, -1))[0])
        phi = np.zeros(self.n, dtype=float)

        # Cache v(S) for all 2^n subsets
        cache = {}
        for r in range(self.n + 1):
            for sub in itertools.combinations(self.features, r):
                S = frozenset(sub)
                cache[S] = self._v(S, x, M)

        # Exact Shapley summation
        for idx, feat in enumerate(self.features):
            rest = [f for f in self.features if f != feat]
            for r in range(self.n):
                for sub in itertools.combinations(rest, r):
                    S     = frozenset(sub)
                    phi[idx] += (
                        self._kw[len(sub)]
                        * self.gamma[feat]
                        * (cache[frozenset(set(sub) | {feat})] - cache[S])
                    )

        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame,
                       k: int, M: int) -> pd.Series:
        k        = min(k, len(X_test))
        phi_list = []
        for i in range(k):
            if (i + 1) % 20 == 0:
                print(f"      instance {i+1}/{k} ...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Metric computation
def compute_metrics(gamma: Dict[str, float],
                    global_abs: pd.Series,
                    features: List[str]) -> Tuple[dict, dict]:
    gt_vec   = np.array([gamma[f]             for f in features])
    raw      = np.array([global_abs.get(f, 0) for f in features])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pr, _  = pearsonr(shap_vec, gt_vec)
    sr, _  = spearmanr(shap_vec, gt_vec)
    cos    = float(np.dot(shap_vec, gt_vec) /
                   (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse   = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae    = float(np.mean(np.abs(shap_vec - gt_vec)))

    metrics = {
        'Pearson_r':  round(float(pr),  4),
        'Spearman_r': round(float(sr),  4),
        'Cosine':     round(cos,        4),
        'RMSE':       round(rmse,       4),
        'MAE':        round(mae,        4),
    }
    attrs = {f"Attr_{f}": round(shap_vec[i], 4)
             for i, f in enumerate(features)}
    return metrics, attrs


#  GAMMA VARIANT BUILDER  (feature -> target importance perturbations)
def build_gamma_variants(true_gamma: Dict[str, float],
                         features: List[str]) -> List[dict]:
    """
    Returns list of dicts:  {label, gamma}
    All returned gamma dicts are renormalised to sum to 1.
    """
    def normalise(g: Dict[str, float]) -> Dict[str, float]:
        g = {f: max(v, 0.0) for f, v in g.items()}   # clip negatives
        tot = sum(g.values())
        if tot < 1e-12:
            return {f: 1.0 / len(features) for f in features}
        return {f: v / tot for f, v in g.items()}

    n   = len(features)
    tg  = true_gamma
    by_val   = sorted(features, key=lambda f: tg[f], reverse=True)
    top_feat = by_val[0]
    bot_feat = by_val[-1]

    variants = [
        # G00 - true gamma (baseline)
        dict(label='G00: true gamma (baseline)',
             gamma=dict(tg)),

        # G01 - uniform
        dict(label='G01: uniform gamma',
             gamma=normalise({f: 1.0 / n for f in features})),

        # G02 - top-1 dominant (all weight on highest-γ feature)
        dict(label=f'G02: top-1 dominant ({top_feat})',
             gamma=normalise({f: (1.0 if f == top_feat else 0.0) for f in features})),

        # G03 - inverted rank (highest -> lowest assigned)
        dict(label='G03: inverted gamma',
             gamma=normalise({f: tg[by_val[n - 1 - i]]
                              for i, f in enumerate(by_val)})),

        # G04 - scale top feature ×2 (renormalise)
        dict(label=f'G04: top feature ({top_feat}) ×2',
             gamma=normalise({f: (tg[f] * 2 if f == top_feat else tg[f])
                              for f in features})),

        # G05 - scale bottom feature ×5 (renormalise)
        dict(label=f'G05: bottom feature ({bot_feat}) ×5',
             gamma=normalise({f: (tg[f] * 5 if f == bot_feat else tg[f])
                              for f in features})),

        # G06 - zero top feature (renormalise over rest)
        dict(label=f'G06: zero top feature ({top_feat})',
             gamma=normalise({f: (0.0 if f == top_feat else tg[f])
                              for f in features})),

        # G07 - zero bottom feature (renormalise over rest)
        dict(label=f'G07: zero bottom feature ({bot_feat})',
             gamma=normalise({f: (0.0 if f == bot_feat else tg[f])
                              for f in features})),

        # G08 - Gaussian noise σ=0.1
        dict(label='G08: gamma + noise σ=0.1',
             gamma=normalise({f: tg[f] + float(rng_noise.normal(0, 0.1))
                              for f in features})),

        # G09 - Gaussian noise σ=0.3
        dict(label='G09: gamma + noise σ=0.3',
             gamma=normalise({f: tg[f] + float(rng_noise.normal(0, 0.3))
                              for f in features})),
    ]
    return variants


# Sem weight variant builder
def build_sem_variants(fitted_sem: LinearMechanismsBatch,
                       true_edges: List[Tuple[str, str]]) -> List[dict]:
    """
    Returns list of dicts: {label, patches}
    patches: Dict[(parent, child) -> new_β]
    Empty patches = baseline (no change).
    Only makes sense when true_edges is non-empty.
    """
    if not true_edges:
        return [dict(label='W00: baseline (no inter-feature edges)', patches={})]

    def β(parent: str, child: str) -> float:
        m   = fitted_sem.mech[child]
        idx = m['parents'].index(parent)
        return float(m['coef'][idx])

    edge_betas = {(p, c): β(p, c) for p, c in true_edges}
    print("\n  True fitted SEM edge weights:")
    for (p, c), val in edge_betas.items():
        print(f"    {p} → {c}:  β = {val:.6f}")

    # For datasets with exactly 2 edges (soil), build single-edge +
    # compound variants. For datasets with 1 edge, compound = same.
    e = list(true_edges)

    variants = [dict(label='W00: baseline (true SEM weights)', patches={})]

    # Single-edge perturbations for each edge
    for p, c in e:
        b = edge_betas[(p, c)]
        tag = f"{p[:3]}→{c[:3]}"
        variants += [
            dict(label=f'W: {tag} ×2',
                 patches={(p, c): b * 2}),
            dict(label=f'W: {tag} ×5',
                 patches={(p, c): b * 5}),
            dict(label=f'W: {tag} ×0.5',
                 patches={(p, c): b * 0.5}),
            dict(label=f'W: {tag} ×0.1',
                 patches={(p, c): b * 0.1}),
            dict(label=f'W: {tag} = 0',
                 patches={(p, c): 0.0}),
            dict(label=f'W: {tag} negated',
                 patches={(p, c): -b}),
            dict(label=f'W: {tag} + N(0,0.5)',
                 patches={(p, c): b + float(rng_noise.normal(0, 0.5))}),
            dict(label=f'W: {tag} + N(0,2.0)',
                 patches={(p, c): b + float(rng_noise.normal(0, 2.0))}),
        ]

    # Compound variants (all edges simultaneously) - only when ≥2 edges
    if len(e) >= 2:
        b0, b1 = edge_betas[e[0]], edge_betas[e[1]]
        variants += [
            dict(label='W: both edges ×2',
                 patches={e[0]: b0 * 2, e[1]: b1 * 2}),
            dict(label='W: both edges ×0.1',
                 patches={e[0]: b0 * 0.1, e[1]: b1 * 0.1}),
            dict(label='W: edge0 neg. + edge1 ×0.1',
                 patches={e[0]: -b0, e[1]: b1 * 0.1}),
            dict(label='W: both edges = 0 (star DAG)',
                 patches={e[0]: 0.0, e[1]: 0.0}),
        ]

    return variants


# Single dataset runner
def run_dataset(
    dataset_name:  str,
    features:      List[str],
    true_edges:    List[Tuple[str, str]],
    Xtr:           pd.DataFrame,
    Xte:           pd.DataFrame,
    y_train:       np.ndarray,
    true_gamma:    Dict[str, float],
    output_csv:    str,
):
    print(f"\n{'='*90}")
    print(f"DATASET: {dataset_name.upper()}")
    print(f"{'='*90}")
    print(f"  Features ({len(features)}): {', '.join(features)}")
    print(f"  True edges: {true_edges if true_edges else 'none (star DAG)'}")
    print(f"  True gamma:")
    for f, v in true_gamma.items():
        print(f"    {f:<28} γ = {v:.4f}")

    # ── Train model (once per dataset) ───────────────────────────
    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        random_state=CONFIG['RANDOM_SEED'],
        verbosity=0, n_jobs=1, tree_method='hist',
    )
    model.fit(Xtr.values, y_train)

    # ── Fit SEM once on true graph ────────────────────────────────
    dag        = build_dag(features, true_edges)
    fitted_sem = LinearMechanismsBatch(dag, features).fit(Xtr)

    # ── Build variant lists ───────────────────────────────────────
    sem_variants   = build_sem_variants(fitted_sem, true_edges)
    gamma_variants = build_gamma_variants(true_gamma, features)

    total_variants = len(sem_variants) + len(gamma_variants)
    print(f"\n  SEM weight variants : {len(sem_variants)}")
    print(f"  Gamma variants      : {len(gamma_variants)}")
    print(f"  Total               : {total_variants}")

    all_results = []
    t_total     = time.time()
    v_counter   = 0

    print(f"\n── Part A: SEM weight perturbations (γ fixed at true) ──")
    for v in sem_variants:
        v_counter += 1
        print(f"\n[{v_counter:02d}/{total_variants}] {v['label']}")

        sem = deepcopy(fitted_sem)
        if v['patches']:
            sem.patch_weights(v['patches'])
            for (p, c), val in v['patches'].items():
                print(f"    patched {p} → {c}: {val:.6f}")

        t0     = time.time()
        engine = ExactCausalSHAPRegressor(
            model=model, sem=sem, gamma=true_gamma, X_train=Xtr
        )
        g_abs  = engine.explain_global(
            Xte, k=CONFIG['EXACT_K'], M=CONFIG['M_RESAMPLES']
        )
        elapsed = time.time() - t0

        m, attrs = compute_metrics(true_gamma, g_abs, features)
        print(
            f"  Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
            f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f} | "
            f"MAE={m['MAE']:.4f} | {elapsed:.1f}s"
        )
        all_results.append({
            'Variant_Type': 'SEM_weight',
            'Label':        v['label'],
            'Patches':      str(v['patches']),
            'Gamma_used':   'true',
            **m, **attrs,
            'Time_s':       round(elapsed, 1),
        })

    print(f"\n── Part B: Gamma (feature→target) perturbations (SEM fixed) ──")
    for v in gamma_variants:
        v_counter += 1
        print(f"\n[{v_counter:02d}/{total_variants}] {v['label']}")
        print(f"    gamma: { {f: round(g,4) for f,g in v['gamma'].items()} }")

        t0     = time.time()
        engine = ExactCausalSHAPRegressor(
            model=model, sem=fitted_sem,   # unmodified SEM
            gamma=v['gamma'], X_train=Xtr
        )
        g_abs  = engine.explain_global(
            Xte, k=CONFIG['EXACT_K'], M=CONFIG['M_RESAMPLES']
        )
        elapsed = time.time() - t0

        m, attrs = compute_metrics(true_gamma, g_abs, features)
        print(
            f"  Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
            f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f} | "
            f"MAE={m['MAE']:.4f} | {elapsed:.1f}s"
        )
        all_results.append({
            'Variant_Type': 'Gamma',
            'Label':        v['label'],
            'Patches':      'n/a',
            'Gamma_used':   str({f: round(g, 4) for f, g in v['gamma'].items()}),
            **m, **attrs,
            'Time_s':       round(elapsed, 1),
        })

    # ── Save ─────────────────────────────────────────────────────
    df = pd.DataFrame(all_results)
    df.to_csv(output_csv, index=False)

    print(f"\n{'='*115}")
    print(f"RESULTS SUMMARY — {dataset_name.upper()}")
    print(f"{'='*115}")
    attr_cols  = [f"Attr_{f}" for f in features[:3]]   # first 3 features for display
    disp_cols  = ['Variant_Type', 'Label', 'Pearson_r', 'Spearman_r',
                  'Cosine', 'RMSE', 'MAE'] + attr_cols
    print(df[disp_cols].to_string(index=False))
    print(f"\nTotal runtime: {(time.time() - t_total) / 60:.1f} min  |  "
          f"Saved → {output_csv}\n")


# Main
def main():
    np.random.seed(CONFIG['RANDOM_SEED'])

    print("\n" + "=" * 90)
    print("WEIGHT SENSITIVITY EXPERIMENT — BOTH DATASETS")
    print("SEM inter-feature weights  AND  gamma (feature→target) weights")
    print("=" * 90)
    print(f"  MC samples per subset : {CONFIG['M_RESAMPLES']}")
    print(f"  Test instances (K)    : {CONFIG['EXACT_K']}")

    # Soil dataset
    dc       = pd.read_csv(CONFIG['SOIL_DATA_PATH']).rename(columns=SOIL_COL_MAP)
    Xtr_s, Xte_s, ytr_s, _ = train_test_split(
        dc[SOIL_FEATURES], dc['SPI_CONTINUOUS'],
        test_size=CONFIG['TEST_SIZE'],
        random_state=CONFIG['RANDOM_SEED'], shuffle=True,
    )
    scaler_s = StandardScaler()
    Xtr_s    = pd.DataFrame(scaler_s.fit_transform(Xtr_s),
                             columns=SOIL_FEATURES, index=Xtr_s.index)
    Xte_s    = pd.DataFrame(scaler_s.transform(Xte_s),
                             columns=SOIL_FEATURES, index=Xte_s.index)
    soil_gamma = compute_soil_gamma(CONFIG['SOIL_GROUND_TRUTH_PATH'])

    run_dataset(
        dataset_name='Soil',
        features=SOIL_FEATURES,
        true_edges=SOIL_TRUE_EDGES,
        Xtr=Xtr_s, Xte=Xte_s,
        y_train=ytr_s.values,
        true_gamma=soil_gamma,
        output_csv='soil_weight_sensitivity_results.csv',
    )

    # Yield dataset
    data     = np.load(CONFIG['YIELD_DATA_PATH'], allow_pickle=True)
    Xtr_y    = pd.DataFrame(data['X_train'], columns=YIELD_FEATURES)
    Xte_y    = pd.DataFrame(data['X_test'],  columns=YIELD_FEATURES)
    ytr_y    = data['y_train'].astype(float)

    scaler_y = StandardScaler()
    Xtr_y    = pd.DataFrame(scaler_y.fit_transform(Xtr_y),
                             columns=YIELD_FEATURES, index=Xtr_y.index)
    Xte_y    = pd.DataFrame(scaler_y.transform(Xte_y),
                             columns=YIELD_FEATURES, index=Xte_y.index)
    yield_gamma = compute_yield_gamma(
        pd.DataFrame(data['X_train'], columns=YIELD_FEATURES)   # unscaled for log()
    )

    run_dataset(
        dataset_name='Yield',
        features=YIELD_FEATURES,
        true_edges=YIELD_TRUE_EDGES,
        Xtr=Xtr_y, Xte=Xte_y,
        y_train=ytr_y,
        true_gamma=yield_gamma,
        output_csv='yield_weight_sensitivity_results.csv',
    )

    print("\nAll done.")


if __name__ == '__main__':
    main()


WEIGHT SENSITIVITY EXPERIMENT — BOTH DATASETS
SEM inter-feature weights  AND  gamma (feature→target) weights
  MC samples per subset : 128
  Test instances (K)    : 100

DATASET: SOIL
  Features (7): Organic Carbon, Nitrogen, Phosphorus, Potassium, pH, Zinc, Electrical Conductivity
  True edges: [('Organic Carbon', 'pH'), ('pH', 'Zinc')]
  True gamma:
    Organic Carbon               γ = 0.0622
    Nitrogen                     γ = 0.4012
    Phosphorus                   γ = 0.0762
    Potassium                    γ = 0.2588
    pH                           γ = 0.0886
    Zinc                         γ = 0.0147
    Electrical Conductivity      γ = 0.0984

  True fitted SEM edge weights:
    Organic Carbon → pH:  β = -0.708307
    pH → Zinc:  β = -0.194760

  SEM weight variants : 21
  Gamma variants      : 10
  Total               : 31

── Part A: SEM weight perturbations (γ fixed at true) ──

[01/31] W00: baseline (true SEM weights)
      instance 20/100 ...
      instance 40/100 ...


In [3]:
"""
Weight Sensitivity Experiment — Causal SHAP on Soil & Yield Regression
=======================================================================
Fixes the TRUE causal graph topology for each dataset and perturbs:

  (A) SEM inter-feature edge weights  (β coefficients in the SEM)
  (B) Feature→target importance weights (γᵢ used inside Causal SHAP)

This complements the graph-structure experiments (G00–G08 / G00–G10)
which varied topology while keeping weights fixed. Here topology is
fixed and weights vary — isolating weight misspecification as the
sole source of attribution error.

DATASETS
--------
  Soil  : data_complete.csv + ground_truth.csv
          True graph edges: OC→pH, pH→Zn
          Features: Organic Carbon, Nitrogen, Phosphorus, Potassium,
                    pH, Zinc, Electrical Conductivity  (7 features)

  Yield : yield_regression_preprocessed.npz
          True graph edges: none (star DAG — all features independent)
          Features: Nitrogen, Irrigation, Precipitation, Herbicide (4 features)

VARIANT FAMILIES  (applied to each dataset independently)
-----------------
  SEM weight variants  — only meaningful where inter-feature edges exist
    W00  : baseline (true fitted weights)
    W01–W04 : scale single edge ×2, ×5, ×0.5, ×0.1
    W05–W06 : zero each edge individually
    W07–W08 : negate each edge individually
    W09–W10 : Gaussian noise σ=0.5 / σ=2.0 on each edge
    W11–W14 : compound (both edges): ×2, ×0.1, neg+shrink, both=0

  Gamma (γᵢ) variants  — feature→target importance misspecification
    G00  : baseline (true γ)
    G01  : uniform γ (all features equally weighted)
    G02  : top-1 dominant (all weight on highest-γ feature)
    G03  : inverted γ (highest true → lowest assigned, and vice versa)
    G04  : scale highest-γ feature ×2 (renormalised)
    G05  : scale lowest-γ feature ×5 (renormalised)
    G06  : zero highest-γ feature (renormalised over rest)
    G07  : zero lowest-γ feature (renormalised over rest)
    G08  : add Gaussian noise σ=0.1 (renormalised, clipped ≥ 0)
    G09  : add Gaussian noise σ=0.3 (renormalised, clipped ≥ 0)

Outputs
-------
    soil_weight_sensitivity_results.csv
    yield_weight_sensitivity_results.csv
"""

import time
import itertools
import math
import warnings
from copy import deepcopy
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import networkx as nx
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")

# Global config
CONFIG = {
    # Soil paths
    'SOIL_DATA_PATH':         '../../datasets/data_complete.csv',
    'SOIL_GROUND_TRUTH_PATH': '../../datasets/ground_truth.csv',
    # Yield path
    'YIELD_DATA_PATH':        '../../datasets/yield_regression_preprocessed.npz',

    'TEST_SIZE':     0.20,
    'M_RESAMPLES':   128,   # MC samples per subset evaluation
    'EXACT_K':       100,   # Test instances evaluated per variant
    'RANDOM_SEED':   42,
}

# ── Soil ──────────────────────────────────────────────────────────
SOIL_FEATURES = [
    'Organic Carbon', 'Nitrogen', 'Phosphorus', 'Potassium',
    'pH', 'Zinc', 'Electrical Conductivity',
]
SOIL_COL_MAP = {
    'OC': 'Organic Carbon', 'N': 'Nitrogen',  'P': 'Phosphorus',
    'K':  'Potassium',      'PH': 'pH',        'ZN': 'Zinc',
    'EC': 'Electrical Conductivity',
}
SOIL_TRUE_EDGES = [('Organic Carbon', 'pH'), ('pH', 'Zinc')]

# ── Yield ─────────────────────────────────────────────────────────
YIELD_FEATURES = ['Nitrogen', 'Irrigation', 'Precipitation', 'Herbicide']
YIELD_TRUE_EDGES: List[Tuple[str, str]] = []   # star DAG - no inter-feature edges

YIELD_TRANSLOG = {
    'betaN':  0.35,  'betaI':  0.50,  'betaP':  0.25,  'betaH':  0.05,
    'betaNN': -0.02, 'betaII': -0.03, 'betaPP': -0.01, 'betaHH': -0.005,
    'betaNI':  0.04, 'betaNP':  0.02, 'betaNH':  0.01,
    'betaIP':  0.03, 'betaIH':  0.01, 'betaPH':  0.005,
}

rng_noise = np.random.default_rng(0)


# Ground-truth gamma functions
def compute_soil_gamma(ground_truth_path: str) -> Dict[str, float]:
    gt  = pd.read_csv(ground_truth_path)
    net = pd.DataFrame({
        'Organic Carbon':          gt['OC_base']  - gt['OC_penalty'],
        'Nitrogen':                gt['N_base']   - gt['N_penalty'],
        'Phosphorus':              gt['P_base']   - gt['P_penalty'],
        'Potassium':               gt['K_base']   - gt['K_penalty'],
        'pH':                     -gt['PH_penalty'],
        'Zinc':                    gt['ZN_base']  - gt['ZN_penalty'],
        'Electrical Conductivity': -gt['EC_penalty'],
    })
    mean_abs = net.abs().mean()
    return (mean_abs / mean_abs.sum()).to_dict()


def compute_yield_gamma(X_train: pd.DataFrame) -> Dict[str, float]:
    # Guard: translog requires log(x) so values must be > 0. Always pass
    # RAW (unscaled) X_train - StandardScaler output breaks this silently.
    for f in YIELD_FEATURES:
        if (X_train[f] <= 0).any():
            raise ValueError(
                f"compute_yield_gamma: non-positive values in '{f}'. "
                "Pass unscaled X_train — StandardScaler output is invalid here."
            )
    c   = YIELD_TRANSLOG
    lm  = {f: float(np.log(X_train[f]).mean()) for f in YIELD_FEATURES}
    lnN, lnI, lnP, lnH = (lm['Nitrogen'], lm['Irrigation'],
                           lm['Precipitation'], lm['Herbicide'])
    elasticities = {
        'Nitrogen':
            c['betaN']  + c['betaNN']*lnN + c['betaNI']*lnI
            + c['betaNP']*lnP + c['betaNH']*lnH,
        'Irrigation':
            c['betaI']  + c['betaII']*lnI + c['betaNI']*lnN
            + c['betaIP']*lnP + c['betaIH']*lnH,
        'Precipitation':
            c['betaP']  + c['betaPP']*lnP + c['betaNP']*lnN
            + c['betaIP']*lnI + c['betaPH']*lnH,
        'Herbicide':
            c['betaH']  + c['betaHH']*lnH + c['betaNH']*lnN
            + c['betaIH']*lnI + c['betaPH']*lnP,
    }
    total = sum(elasticities.values())
    return {f: v / total for f, v in elasticities.items()}


# Dag builder
def build_dag(features: List[str],
              inter_edges: List[Tuple[str, str]]) -> nx.DiGraph:
    G = nx.DiGraph()
    G.add_nodes_from(features)
    for u, v in inter_edges:
        G.add_edge(u, v)
    if not nx.is_directed_acyclic_graph(G):
        raise ValueError(f"Cycle detected: {inter_edges}")
    return G


class LinearMechanismsBatch:
    """
    Linear-Gaussian SEM fitted on a given DAG.
    `patch_weights()` overrides specific β coefficients in-place
    without re-running OLS — isolating weight misspecification.
    """
    def __init__(self, dag: nx.DiGraph, features: List[str]):
        self.dag      = dag
        self.features = features
        self.mech:    Dict[str, dict] = {}
        self.topo:    List[str] = list(nx.topological_sort(dag))

    def fit(self, X: pd.DataFrame) -> 'LinearMechanismsBatch':
        for var in self.features:
            parents = list(self.dag.predecessors(var))
            if not parents:
                self.mech[var] = {
                    'parents':   [],
                    'intercept': float(X[var].mean()),
                    'coef':      None,
                    'sigma':     float(X[var].std(ddof=0)),
                }
            else:
                lr    = LinearRegression().fit(X[parents].values, X[var].values)
                resid = X[var].values - lr.predict(X[parents].values)
                self.mech[var] = {
                    'parents':   parents,
                    'intercept': float(lr.intercept_),
                    'coef':      lr.coef_.astype(float).copy(),
                    'sigma':     float(np.std(resid, ddof=0)),
                }
        return self

    def patch_weights(self,
                      patches: Dict[Tuple[str, str], float]
                      ) -> 'LinearMechanismsBatch':
        """patches: {(parent, child): new_β_value}"""
        for (parent, child), val in patches.items():
            m   = self.mech[child]
            idx = m['parents'].index(parent)
            m['coef'][idx] = float(val)
        return self

    def sample_intervention_batch(
        self, S: Set[str], x_obs: pd.Series, M: int, rng_inst
    ) -> np.ndarray:
        X_out       = np.empty((M, len(self.features)))
        feat_to_idx = {f: i for i, f in enumerate(self.features)}
        simulated   = {}

        for var in self.topo:
            v_idx = feat_to_idx[var]
            if var in S:
                val = float(x_obs[var])
                X_out[:, v_idx] = val
                simulated[var]  = np.full(M, val)
                continue
            m = self.mech[var]
            if not m['parents']:
                mu = np.full(M, m['intercept'])
            else:
                pm = np.empty((M, len(m['parents'])))
                for pi, p in enumerate(m['parents']):
                    pm[:, pi] = simulated[p] if p in simulated else float(x_obs[p])
                mu = m['intercept'] + pm @ m['coef']
            X_out[:, v_idx] = (
                mu if m['sigma'] < 1e-8
                else rng_inst.normal(mu, m['sigma'], size=M)
            )
            simulated[var] = X_out[:, v_idx]
        return X_out


# Exact causal shap engine
class ExactCausalSHAPRegressor:
    def __init__(self,
                 model,
                 sem:     LinearMechanismsBatch,
                 gamma:   Dict[str, float],
                 X_train: pd.DataFrame):
        self.model      = model
        self.features   = list(X_train.columns)
        self.n          = len(self.features)
        self.gamma      = gamma        # caller is responsible for normalisation
        self.sem        = sem
        self._Ef        = float(model.predict(X_train.values).mean())
        self._rng       = np.random.default_rng(CONFIG['RANDOM_SEED'])

        # Shapley kernel weights keyed by coalition size
        self._kw = {}
        for sz in range(self.n):
            num = math.factorial(sz) * math.factorial(self.n - sz - 1)
            self._kw[sz] = float(num / math.factorial(self.n))

    def _v(self, S: frozenset, x: pd.Series, M: int) -> float:
        X_mat = self.sem.sample_intervention_batch(S, x, M, self._rng)
        return float(np.mean(self.model.predict(X_mat)))

    def explain_instance(self, x: pd.Series, M: int) -> pd.Series:
        fx  = float(self.model.predict(x.values.reshape(1, -1))[0])
        phi = np.zeros(self.n, dtype=float)

        # Cache v(S) for all 2^n subsets
        cache = {}
        for r in range(self.n + 1):
            for sub in itertools.combinations(self.features, r):
                S = frozenset(sub)
                cache[S] = self._v(S, x, M)

        # Exact Shapley summation
        for idx, feat in enumerate(self.features):
            rest = [f for f in self.features if f != feat]
            for r in range(self.n):
                for sub in itertools.combinations(rest, r):
                    S     = frozenset(sub)
                    phi[idx] += (
                        self._kw[len(sub)]
                        * self.gamma[feat]
                        * (cache[frozenset(set(sub) | {feat})] - cache[S])
                    )

        s = float(phi.sum())
        if abs(s) > 1e-12:
            phi = phi * ((fx - self._Ef) / s)
        return pd.Series(phi, index=self.features)

    def explain_global(self, X_test: pd.DataFrame,
                       k: int, M: int) -> pd.Series:
        k        = min(k, len(X_test))
        phi_list = []
        for i in range(k):
            if (i + 1) % 20 == 0:
                print(f"      instance {i+1}/{k} ...", flush=True)
            phi_list.append(self.explain_instance(X_test.iloc[i], M))
        return pd.concat(phi_list, axis=1).abs().mean(axis=1)


# Metric computation
def compute_metrics(gamma: Dict[str, float],
                    global_abs: pd.Series,
                    features: List[str]) -> Tuple[dict, dict]:
    gt_vec   = np.array([gamma[f]             for f in features])
    raw      = np.array([global_abs.get(f, 0) for f in features])
    shap_vec = raw / raw.sum() if raw.sum() > 1e-12 else raw

    pr, _  = pearsonr(shap_vec, gt_vec)
    sr, _  = spearmanr(shap_vec, gt_vec)
    cos    = float(np.dot(shap_vec, gt_vec) /
                   (np.linalg.norm(shap_vec) * np.linalg.norm(gt_vec) + 1e-12))
    rmse   = float(np.sqrt(np.mean((shap_vec - gt_vec) ** 2)))
    mae    = float(np.mean(np.abs(shap_vec - gt_vec)))

    metrics = {
        'Pearson_r':  round(float(pr),  4),
        'Spearman_r': round(float(sr),  4),
        'Cosine':     round(cos,        4),
        'RMSE':       round(rmse,       4),
        'MAE':        round(mae,        4),
    }
    attrs = {f"Attr_{f}": round(shap_vec[i], 4)
             for i, f in enumerate(features)}
    return metrics, attrs


#  GAMMA VARIANT BUILDER  (feature -> target importance perturbations)
def build_gamma_variants(true_gamma: Dict[str, float],
                         features: List[str]) -> List[dict]:
    """
    Returns list of dicts:  {label, gamma}
    All returned gamma dicts are renormalised to sum to 1.
    """
    def normalise(g: Dict[str, float]) -> Dict[str, float]:
        g = {f: max(v, 0.0) for f, v in g.items()}   # clip negatives
        tot = sum(g.values())
        if tot < 1e-12:
            return {f: 1.0 / len(features) for f in features}
        return {f: v / tot for f, v in g.items()}

    n   = len(features)
    tg  = true_gamma
    by_val   = sorted(features, key=lambda f: tg[f], reverse=True)
    top_feat = by_val[0]
    bot_feat = by_val[-1]

    variants = [
        # G00 - true gamma (baseline)
        dict(label='G00: true gamma (baseline)',
             gamma=dict(tg)),

        # G01 - uniform
        dict(label='G01: uniform gamma',
             gamma=normalise({f: 1.0 / n for f in features})),

        # G02 - top-1 dominant (all weight on highest-γ feature)
        dict(label=f'G02: top-1 dominant ({top_feat})',
             gamma=normalise({f: (1.0 if f == top_feat else 0.0) for f in features})),

        # G03 - inverted rank (highest -> lowest assigned)
        dict(label='G03: inverted gamma',
             gamma=normalise({f: tg[by_val[n - 1 - i]]
                              for i, f in enumerate(by_val)})),

        # G04 - scale top feature ×2 (renormalise)
        dict(label=f'G04: top feature ({top_feat}) ×2',
             gamma=normalise({f: (tg[f] * 2 if f == top_feat else tg[f])
                              for f in features})),

        # G05 - scale bottom feature ×5 (renormalise)
        dict(label=f'G05: bottom feature ({bot_feat}) ×5',
             gamma=normalise({f: (tg[f] * 5 if f == bot_feat else tg[f])
                              for f in features})),

        # G06 - zero top feature (renormalise over rest)
        dict(label=f'G06: zero top feature ({top_feat})',
             gamma=normalise({f: (0.0 if f == top_feat else tg[f])
                              for f in features})),

        # G07 - zero bottom feature (renormalise over rest)
        dict(label=f'G07: zero bottom feature ({bot_feat})',
             gamma=normalise({f: (0.0 if f == bot_feat else tg[f])
                              for f in features})),

        # G08 - Gaussian noise σ=0.1
        dict(label='G08: gamma + noise σ=0.1',
             gamma=normalise({f: tg[f] + float(rng_noise.normal(0, 0.1))
                              for f in features})),

        # G09 - Gaussian noise σ=0.3
        dict(label='G09: gamma + noise σ=0.3',
             gamma=normalise({f: tg[f] + float(rng_noise.normal(0, 0.3))
                              for f in features})),
    ]
    return variants


# Sem weight variant builder
def build_sem_variants(fitted_sem: LinearMechanismsBatch,
                       true_edges: List[Tuple[str, str]]) -> List[dict]:
    """
    Returns list of dicts: {label, patches}
    patches: Dict[(parent, child) -> new_β]
    Empty patches = baseline (no change).
    Only makes sense when true_edges is non-empty.
    """
    if not true_edges:
        return [dict(label='W00: baseline (no inter-feature edges)', patches={})]

    def β(parent: str, child: str) -> float:
        m   = fitted_sem.mech[child]
        idx = m['parents'].index(parent)
        return float(m['coef'][idx])

    edge_betas = {(p, c): β(p, c) for p, c in true_edges}
    print("\n  True fitted SEM edge weights:")
    for (p, c), val in edge_betas.items():
        print(f"    {p} → {c}:  β = {val:.6f}")

    # For datasets with exactly 2 edges (soil), build single-edge +
    # compound variants. For datasets with 1 edge, compound = same.
    e = list(true_edges)

    variants = [dict(label='W00: baseline (true SEM weights)', patches={})]

    # Single-edge perturbations for each edge
    for p, c in e:
        b = edge_betas[(p, c)]
        tag = f"{p[:3]}→{c[:3]}"
        variants += [
            dict(label=f'W: {tag} ×2',
                 patches={(p, c): b * 2}),
            dict(label=f'W: {tag} ×5',
                 patches={(p, c): b * 5}),
            dict(label=f'W: {tag} ×0.5',
                 patches={(p, c): b * 0.5}),
            dict(label=f'W: {tag} ×0.1',
                 patches={(p, c): b * 0.1}),
            dict(label=f'W: {tag} = 0',
                 patches={(p, c): 0.0}),
            dict(label=f'W: {tag} negated',
                 patches={(p, c): -b}),
            dict(label=f'W: {tag} + N(0,0.5)',
                 patches={(p, c): b + float(rng_noise.normal(0, 0.5))}),
            dict(label=f'W: {tag} + N(0,2.0)',
                 patches={(p, c): b + float(rng_noise.normal(0, 2.0))}),
        ]

    # Compound variants (all edges simultaneously) - only when ≥2 edges
    if len(e) >= 2:
        b0, b1 = edge_betas[e[0]], edge_betas[e[1]]
        variants += [
            dict(label='W: both edges ×2',
                 patches={e[0]: b0 * 2, e[1]: b1 * 2}),
            dict(label='W: both edges ×0.1',
                 patches={e[0]: b0 * 0.1, e[1]: b1 * 0.1}),
            dict(label='W: edge0 neg. + edge1 ×0.1',
                 patches={e[0]: -b0, e[1]: b1 * 0.1}),
            dict(label='W: both edges = 0 (star DAG)',
                 patches={e[0]: 0.0, e[1]: 0.0}),
        ]

    return variants


# Single dataset runner
def run_dataset(
    dataset_name:  str,
    features:      List[str],
    true_edges:    List[Tuple[str, str]],
    Xtr:           pd.DataFrame,
    Xte:           pd.DataFrame,
    y_train:       np.ndarray,
    true_gamma:    Dict[str, float],
    output_csv:    str,
):
    print(f"\n{'='*90}")
    print(f"DATASET: {dataset_name.upper()}")
    print(f"{'='*90}")
    print(f"  Features ({len(features)}): {', '.join(features)}")
    print(f"  True edges: {true_edges if true_edges else 'none (star DAG)'}")
    print(f"  True gamma:")
    for f, v in true_gamma.items():
        print(f"    {f:<28} γ = {v:.4f}")

    # ── Train model (once per dataset) ───────────────────────────
    model = XGBRegressor(
        n_estimators=160, max_depth=5, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        random_state=CONFIG['RANDOM_SEED'],
        verbosity=0, n_jobs=1, tree_method='hist',
    )
    model.fit(Xtr.values, y_train)

    # ── Fit SEM once on true graph ────────────────────────────────
    dag        = build_dag(features, true_edges)
    fitted_sem = LinearMechanismsBatch(dag, features).fit(Xtr)

    # ── Build variant lists ───────────────────────────────────────
    sem_variants   = build_sem_variants(fitted_sem, true_edges)
    gamma_variants = build_gamma_variants(true_gamma, features)

    total_variants = len(sem_variants) + len(gamma_variants)
    print(f"\n  SEM weight variants : {len(sem_variants)}")
    print(f"  Gamma variants      : {len(gamma_variants)}")
    print(f"  Total               : {total_variants}")

    all_results = []
    t_total     = time.time()
    v_counter   = 0

    print(f"\n── Part A: SEM weight perturbations (γ fixed at true) ──")
    for v in sem_variants:
        v_counter += 1
        print(f"\n[{v_counter:02d}/{total_variants}] {v['label']}")

        sem = deepcopy(fitted_sem)
        if v['patches']:
            sem.patch_weights(v['patches'])
            for (p, c), val in v['patches'].items():
                print(f"    patched {p} → {c}: {val:.6f}")

        t0     = time.time()
        engine = ExactCausalSHAPRegressor(
            model=model, sem=sem, gamma=true_gamma, X_train=Xtr
        )
        g_abs  = engine.explain_global(
            Xte, k=CONFIG['EXACT_K'], M=CONFIG['M_RESAMPLES']
        )
        elapsed = time.time() - t0

        m, attrs = compute_metrics(true_gamma, g_abs, features)
        print(
            f"  Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
            f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f} | "
            f"MAE={m['MAE']:.4f} | {elapsed:.1f}s"
        )
        all_results.append({
            'Variant_Type': 'SEM_weight',
            'Label':        v['label'],
            'Patches':      str(v['patches']),
            'Gamma_used':   'true',
            **m, **attrs,
            'Time_s':       round(elapsed, 1),
        })

    print(f"\n── Part B: Gamma (feature→target) perturbations (SEM fixed) ──")
    for v in gamma_variants:
        v_counter += 1
        print(f"\n[{v_counter:02d}/{total_variants}] {v['label']}")
        print(f"    gamma: { {f: round(g,4) for f,g in v['gamma'].items()} }")

        t0     = time.time()
        engine = ExactCausalSHAPRegressor(
            model=model, sem=fitted_sem,   # unmodified SEM
            gamma=v['gamma'], X_train=Xtr
        )
        g_abs  = engine.explain_global(
            Xte, k=CONFIG['EXACT_K'], M=CONFIG['M_RESAMPLES']
        )
        elapsed = time.time() - t0

        m, attrs = compute_metrics(true_gamma, g_abs, features)
        print(
            f"  Pearson={m['Pearson_r']:.4f} | Spearman={m['Spearman_r']:.4f} | "
            f"Cosine={m['Cosine']:.4f} | RMSE={m['RMSE']:.4f} | "
            f"MAE={m['MAE']:.4f} | {elapsed:.1f}s"
        )
        all_results.append({
            'Variant_Type': 'Gamma',
            'Label':        v['label'],
            'Patches':      'n/a',
            'Gamma_used':   str({f: round(g, 4) for f, g in v['gamma'].items()}),
            **m, **attrs,
            'Time_s':       round(elapsed, 1),
        })

    # ── Save ─────────────────────────────────────────────────────
    df = pd.DataFrame(all_results)
    df.to_csv(output_csv, index=False)

    print(f"\n{'='*115}")
    print(f"RESULTS SUMMARY — {dataset_name.upper()}")
    print(f"{'='*115}")
    attr_cols  = [f"Attr_{f}" for f in features[:3]]   # first 3 features for display
    disp_cols  = ['Variant_Type', 'Label', 'Pearson_r', 'Spearman_r',
                  'Cosine', 'RMSE', 'MAE'] + attr_cols
    print(df[disp_cols].to_string(index=False))
    print(f"\nTotal runtime: {(time.time() - t_total) / 60:.1f} min  |  "
          f"Saved → {output_csv}\n")


# Main
def main():
    np.random.seed(CONFIG['RANDOM_SEED'])

    print("\n" + "=" * 90)
    print("WEIGHT SENSITIVITY EXPERIMENT — BOTH DATASETS")
    print("SEM inter-feature weights  AND  gamma (feature→target) weights")
    print("=" * 90)
    print(f"  MC samples per subset : {CONFIG['M_RESAMPLES']}")
    print(f"  Test instances (K)    : {CONFIG['EXACT_K']}")

    # Soil dataset
    dc       = pd.read_csv(CONFIG['SOIL_DATA_PATH']).rename(columns=SOIL_COL_MAP)
    Xtr_s, Xte_s, ytr_s, _ = train_test_split(
        dc[SOIL_FEATURES], dc['SPI_CONTINUOUS'],
        test_size=CONFIG['TEST_SIZE'],
        random_state=CONFIG['RANDOM_SEED'], shuffle=True,
    )
    scaler_s = StandardScaler()
    Xtr_s    = pd.DataFrame(scaler_s.fit_transform(Xtr_s),
                             columns=SOIL_FEATURES, index=Xtr_s.index)
    Xte_s    = pd.DataFrame(scaler_s.transform(Xte_s),
                             columns=SOIL_FEATURES, index=Xte_s.index)
    soil_gamma = compute_soil_gamma(CONFIG['SOIL_GROUND_TRUTH_PATH'])

    run_dataset(
        dataset_name='Soil',
        features=SOIL_FEATURES,
        true_edges=SOIL_TRUE_EDGES,
        Xtr=Xtr_s, Xte=Xte_s,
        y_train=ytr_s.values,
        true_gamma=soil_gamma,
        output_csv='soil_weight_sensitivity_results.csv',
    )

    # Yield dataset
    data          = np.load(CONFIG['YIELD_DATA_PATH'], allow_pickle=True)
    Xtr_y_raw     = pd.DataFrame(data['X_train'], columns=YIELD_FEATURES)
    Xte_y_raw     = pd.DataFrame(data['X_test'],  columns=YIELD_FEATURES)
    ytr_y         = data['y_train'].astype(float)

    # Gamma MUST be computed from unscaled data - translog uses log(x),
    # which is undefined / wrong on standardised (zero-mean) values.
    yield_gamma = compute_yield_gamma(Xtr_y_raw)

    # Scale AFTER computing gamma
    scaler_y = StandardScaler()
    Xtr_y    = pd.DataFrame(scaler_y.fit_transform(Xtr_y_raw),
                             columns=YIELD_FEATURES, index=Xtr_y_raw.index)
    Xte_y    = pd.DataFrame(scaler_y.transform(Xte_y_raw),
                             columns=YIELD_FEATURES, index=Xte_y_raw.index)

    run_dataset(
        dataset_name='Yield',
        features=YIELD_FEATURES,
        true_edges=YIELD_TRUE_EDGES,
        Xtr=Xtr_y, Xte=Xte_y,
        y_train=ytr_y,
        true_gamma=yield_gamma,
        output_csv='yield_weight_sensitivity_results.csv',
    )

    print("\nAll done.")


if __name__ == '__main__':
    main()


WEIGHT SENSITIVITY EXPERIMENT — BOTH DATASETS
SEM inter-feature weights  AND  gamma (feature→target) weights
  MC samples per subset : 128
  Test instances (K)    : 100

DATASET: SOIL
  Features (7): Organic Carbon, Nitrogen, Phosphorus, Potassium, pH, Zinc, Electrical Conductivity
  True edges: [('Organic Carbon', 'pH'), ('pH', 'Zinc')]
  True gamma:
    Organic Carbon               γ = 0.0622
    Nitrogen                     γ = 0.4012
    Phosphorus                   γ = 0.0762
    Potassium                    γ = 0.2588
    pH                           γ = 0.0886
    Zinc                         γ = 0.0147
    Electrical Conductivity      γ = 0.0984

  True fitted SEM edge weights:
    Organic Carbon → pH:  β = -0.708307
    pH → Zinc:  β = -0.194760

  SEM weight variants : 21
  Gamma variants      : 10
  Total               : 31

── Part A: SEM weight perturbations (γ fixed at true) ──

[01/31] W00: baseline (true SEM weights)
      instance 20/100 ...
      instance 40/100 ...
